모델 배포 개론 03  
Last modified : 2026.08   
작성 : 박광석 (모두의연구소)    
수정 : 김지성 (모두의연구소)

## 실습 기록

```
작성자   김민욱
실습일   2026-08-14
환경     Ubuntu · Python 3.12.3 · fastapi 0.115.0 · uvicorn 0.30.0
         torch 2.9.1(ROCm) · CPU 32코어
서버     127.0.0.1 의 8000~8004 를 나눠 썼다
정리 문서 같은 폴더의 DP03_실습정리.md
```

### 이 노트북에서 내가 더한 것

아이펠에서 받은 원본에 아래를 덧붙였다. **원래 있던 셀은 고치지 않았고, 전부 뒤에 붙이는 방식으로 했다.**

**1. 6.3절 뒤에 추가 실험 A 부터 K 까지 (코드 셀 11개 + 설명 마크다운)**

6.2절에서 동시 요청을 1, 2, 4, 8개로 늘려가며 재봤는데 0.02 / 0.01 / 0.02 / 0.03초로 거의 같게 나왔다.
추론 한 건이 6밀리초라 그런 것인데, 그러면 이 표로는 `run_in_executor` 를 붙인 효과가 있는지 없는지 알 수가 없다.
그래서 조건을 바꿔가며 다시 재보게 됐다.

```
A   부하를 300개까지 올려봤다               완전히 직렬로 나왔다
B   서버를 노트북 커널 밖으로 빼봤다          A 의 직렬은 측정 환경 때문이었다
C   run_in_executor 를 뺀 서버와 비교        차이를 못 봤다
D   /health 로 클라이언트 천장을 쟀다         서버가 병목이 된 적이 없었다
E   추론을 ollama 로 바꿔 네 방식 비교        헬스체크가 갈렸다
F   스레드를 몇 개 쓰는지 세봤다              측정에 결함이 있었다
F2  방식마다 서버를 새로 띄워 다시 셌다        표가 깨끗해졌다
G   재현이 안 되던 숫자를 3번씩 다시 쟀다      이상치였다
H   CPU 바운드로 돌아와 결론을 냈다           파이토치가 GIL 을 놓는다
I   max_workers 를 바꿔가며 쟀다            내 가설(파도 모델)이 기각됐다
J   HTTP 를 통째로 빼고 다시 쟀다            천장은 서버가 아니라 파이토치 쪽이었다
K   예외를 세 자리에서 터뜨려봤다             워커는 안 죽었다
```

**2. 실험용 서버 파일 세 개**

원본이 만드는 `app/main_final.py` 말고, 비교하려고 따로 만든 것들이다.

```
app/main_blocking.py   main_final.py 에서 run_in_executor 한 줄만 뺀 대조군 (실험 C)
app/main_llm.py        추론을 ollama 로 넘기는 네 가지 방식 + 스레드 세는 장치 (실험 E·F)
app/main_cpu.py        torch 스레드와 칸 수를 환경변수로 바꿀 수 있는 서버 (실험 H·I·K)
```

**3. 읽을 때 참고**

- 실험 셀 첫 줄의 `# %load ...` 는 코드를 어디서 가져왔는지 적어둔 흔적이고,
  실행에 필요한 코드는 그 아래에 전부 들어 있다.
- 각 실험 코드 맨 위 주석에 그때 무슨 생각으로 짰는지 적어뒀다.
- 앞에서 내린 판단이 뒤에서 뒤집히는 곳이 몇 군데 있는데, 고치지 않고 그대로 뒀다.
  틀린 채로 넘어간 자리와 그걸 알아챈 자리가 같이 있어야 왜 그렇게 갔는지가 남을 것 같았다.

---

In [1]:
# 서버 실행 도우미 — 노트북 맨 처음에 한 번 실행하세요.
# 노트북 안에서 uvicorn 서버를 띄우고 멈추는 함수를 정의합니다.
import os, sys, asyncio, threading, time, socket, contextlib
import uvicorn

# 작업 디렉터리를 app/ 가 있는 위치로 맞춥니다 (notebooks/ 안에서 열어도 동작).
if not os.path.isdir('app') and os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
# 코드를 저장할 폴더를 미리 만들어 둡니다.
for _d in ('app', 'models', 'data', 'frontend'):
    os.makedirs(_d, exist_ok=True)

_SERVERS = {}  # port -> (server, thread)

def _port_open(host, port):
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def stop_server(port=8000):
    """실행 중인 서버를 멈춥니다."""
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host='127.0.0.1', port=8000, log_level='warning'):
    """백그라운드에서 uvicorn 서버를 띄웁니다.

    app: FastAPI 객체 또는 'app.main:app' 같은 import 경로.
    같은 포트에 서버가 이미 있으면 먼저 멈추고 새로 띄웁니다.
    """
    stop_server(port)
    if _port_open(host, port):
        print(f'⚠️ 포트 {port}를 다른 프로세스가 사용 중입니다 (다른 노트북의 서버일 가능성).')
        print('   그 노트북에서 stop_server(8000)을 실행하거나 커널을 종료한 뒤, 이 셀을 다시 실행하세요.')
        return None
    if isinstance(app, str):
        sys.modules.pop(app.split(':')[0], None)   # 파일을 다시 저장한 경우 최신 내용 반영
    for _ in range(50):
        if not _port_open(host, port):
            break
        time.sleep(0.1)
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop='asyncio')
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    def _run():
        # Windows는 SelectorEventLoop, 그 외는 기본 이벤트 루프를 사용합니다.
        if sys.platform == 'win32':
            loop = asyncio.SelectorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    for _ in range(40):
        if _port_open(host, port):
            print(f'서버 실행됨: http://{host}:{port}')
            return server
        time.sleep(0.25)
    print('서버가 시작되지 않았습니다. 위 로그를 확인하세요.')
    return server

print('서버 도우미 준비 완료 (serve_in_thread, stop_server)')

서버 도우미 준비 완료 (serve_in_thread, stop_server)


# Day 3 — 비동기 처리와 에러 핸들링
```
[동기 API 서버 — 모델 추론에 3초 소요]

요청 A (0초) → 추론 → 응답 (3초)
요청 B (0초) → 대기 → 추론 → 응답 (6초)
요청 C (0초) → 대기 → 대기 → 추론 → 응답 (9초)
```

I/O 바운드 vs CPU 바운드:
- I/O 바운드 (DB, 외부 API) → 비동기로 효율 극대화 가능
- **CPU 바운드 (모델 추론)** → 단순 async/await만으로는 해결 안 됨 → `run_in_executor` 필요

`async def` 안에서 동기 작업을 수행하면 이벤트 루프가 멈춰 다른 요청을 처리할 수 없습니다.


## 1. 동기 vs 비동기: 왜 API 서버에서 중요한가?

> **학습 목표**
> - 동기(Synchronous)와 비동기(Asynchronous)의 차이를 명확히 구분할 수 있습니다.
> - 모델 추론 API에서 동기 처리가 왜 심각한 병목이 되는지 설명할 수 있습니다.
> - FastAPI가 내부적으로 요청을 어떻게 처리하는지 흐름을 이해합니다.

### 1.1 지금까지의 흐름 복습

Day 2에서 완성한 API는 정상적으로 동작합니다.
한 명이 요청을 보내면, 정확한 결과가 돌아옵니다.

그런데 **동시에 여러 명이 요청을 보내면** 어떻게 될까요?

이것이 오늘의 핵심 질문입니다.

### 1.2 동기 처리: 한 번에 하나씩

동기(Synchronous) 처리는 **작업을 순서대로, 하나씩** 수행하는 방식입니다.
앞의 작업이 끝나야 다음 작업을 시작합니다.

일상적인 비유로 설명하겠습니다.

![image.png](images/nb/nb_01_ef5dfd89.png)

세 명의 고객(A, B, C)이 차례로 서 있으며, 각각 아메리카노, 라떼, 주스를 주문한 상태입니다.  
한 명의 바리스타가 주문을 하나씩 처리하고 있습니다.  
하단에 시간 표시(0분, 3분, 6분, 9분)는 고객 A는 대기 시간 0분, 고객 B는 3분, 고객 C는 6분 뒤에 음료를 받게 된다는 것을 시각적인 막대로 명확히 보여줍니다.

이것을 API 서버에 대입하면:



```
[동기 API 서버 — 모델 추론에 3초 소요]

요청 A 도착 (0초)  → 추론 시작 → 추론 완료 (3초) → 응답
요청 B 도착 (0초)  → 대기...                      → 추론 시작 (3초) → 응답 (6초)
요청 C 도착 (0초)  → 대기...                       → 대기...        → 추론 시작 (6초) → 응답 (9초)

시간축:
0초 ─────── 3초 ─────── 6초 ─────── 9초
 ███████████                              요청 A (3초 소요)
             ███████████                  요청 B (6초 대기 포함)
                          ███████████     요청 C (9초 대기 포함)
```


> 요청 B와 C는 서버가 바빠서가 아니라, **앞의 요청이 끝나기를 기다리느라** 느려진 것입니다.


### 1.3 비동기 처리: 대기 시간을 활용


비동기(Asynchronous) 처리는 **어떤 작업이 대기 상태일 때, 다른 작업을 처리**하는 방식입니다.


![image.png](images/nb/nb_02_4ab7684e.png)

직원이 A의 주문을 받고 에스프레소 머신을 가동합니다.  
머신이 돌아가는 그 시간 동안, 직원은 B의 주문을 받습니다.  
  
직원은 B의 우유 스티밍을 시작합니다 (B 스티밍 중).  
머신(A 추출 중)과 스티밍(B 스티밍 중)이 동시에 돌아갑니다.  
직원은 C의 주문을 받습니다.  

직원은 C의 블렌더를 가동합니다 (C 블렌딩 중).  
머신(A), 스티머(B), 블렌더(C) 세 기계가 동시에 작동하고, 직원은 세 작업을 오갑니다.  

이렇게 9분이 걸리던 일을 3분만에 수행해내었습니다.  


> 비동기의 핵심은 **직원(CPU)을 늘리는 것이 아니라,
> 대기 시간에 놀지 않고 다른 일을 하는 것**입니다.


### 1.4 API 서버에서 "대기"란 무엇입니까?

웹 서버가 처리하는 작업은 크게 두 종류로 나뉩니다.


![image.png](images/nb/nb_03_ccf568ba.png)


여기서 중요한 점이 있습니다:

> **모델 추론은 CPU 바운드 작업입니다.**

모델이 `forward()` 연산을 수행하는 동안 CPU(또는 GPU)는 쉬지 않고 계산합니다.
이 시간에 "다른 일을 할 수 있는 대기 시간"이 존재하지 않습니다.

그렇다면 비동기가 모델 추론에는 의미가 없을까요?
결론부터 말씀드리면, **의미가 있습니다**. 하지만 단순한 `async/await`만으로는 부족하고,
추가적인 패턴이 필요합니다. 이것이 오늘 섹션 4에서 다룰 `run_in_executor`입니다.


### 1.5 FastAPI의 요청 처리 방식

FastAPI가 내부적으로 요청을 어떻게 처리하는지 이해하면,
"왜 동기 추론이 문제인지"가 명확해집니다.



```
FastAPI는 내부적으로 단일 이벤트 루프(Event Loop)에서 동작합니다.

[이벤트 루프 — 단일 스레드]

  요청 A 도착 → 핸들러 실행 → ... → 응답 반환
  요청 B 도착 → (A가 끝날 때까지 대기) → 핸들러 실행 → ...
```

FastAPI는 함수의 정의 방식에 따라 다르게 동작합니다:


```python
# 방식 1: 일반 함수 (def)
@app.post("/predict")
def predict(request: PredictRequest):
    result = model(input_tensor)    # 이 동안 이벤트 루프가 블로킹되지 않음
    return result                   # FastAPI가 별도 스레드풀에서 실행해줌

# 방식 2: 비동기 함수 (async def)
@app.post("/predict")
async def predict(request: PredictRequest):
    result = model(input_tensor)    # ⚠️ 이 동안 이벤트 루프가 블로킹됨!
    return result                   # 다른 요청을 받을 수 없음
```



> ⚠️ **이것이 오늘의 핵심 함정입니다.**
>
> `async def`로 선언한 함수 안에서 **동기 작업(모델 추론)**을 수행하면,
> 이벤트 루프가 그 작업이 끝날 때까지 멈춥니다.
> 멈춰 있는 동안 다른 요청은 처리되지 않습니다.
>
> 반면 일반 `def`로 선언하면, FastAPI가 자동으로 별도 스레드에서 실행합니다.
> 하지만 이 방식은 스레드 수에 제한이 있어, 최적의 해결책은 아닙니다.

이 문제를 섹션 3에서 직접 재현하고, 섹션 4에서 해결하겠습니다.



### 1.6 정리: 오늘 풀어야 할 문제

 Day 2에서 만든 API는 한 번에 하나의 요청만 처리할 수 있습니다.  
모델 추론이 3초 걸린다면, 10명이 동시에 요청하면 마지막 사용자는 30초를 기다려야 합니다.             

해결 방향  
1. async/await의 원리를 이해한다 (섹션 2)  
2. 문제를 직접 재현한다 (섹션 3)  
3. run_in_executor로 해결한다 (섹션 4)  
4. 에러 핸들링으로 안정성을 높인다 (섹션 5)  
5. 개선 효과를 측정한다 (섹션 6)          

## 2. async/await의 기본 원리



> **학습 목표**
> - `async def`와 `await`의 의미를 코드로 이해합니다.
> - 동기 함수와 비동기 함수의 실행 시간 차이를 직접 비교합니다.

### 2.1 async/await란 무엇입니까?


```python
async def   → "이 함수는 비동기 함수입니다" (중간에 멈췄다가 다시 실행될 수 있음)
await       → "이 작업이 끝날 때까지 기다리되, 그 동안 다른 일을 해도 됩니다"
```

핵심 차이는 `time.sleep()` vs `asyncio.sleep()`입니다:

```
time.sleep(3)          → CPU를 3초간 점유. 그 동안 아무것도 할 수 없음.
await asyncio.sleep(3) → "3초 후에 깨워줘"라고 등록 후, 다른 작업 처리.
```

### 2.2 실습: 동기 vs 비동기 실행 시간 비교


위에 소개한 내용을 직접 확인해보겠습니다.  

In [2]:
import time
import asyncio

In [3]:
# 동기 방식: 순차 실행
def sync_task(name, seconds):
    print(f"  [{name}] 시작")
    time.sleep(seconds)
    print(f"  [{name}] 완료 ({seconds}초)")

print("===== 동기 실행 =====")
start = time.time()
sync_task("작업A", 2)
sync_task("작업B", 2)
sync_task("작업C", 2)
print(f"\n총 소요 시간: {time.time() - start:.1f}초")

===== 동기 실행 =====
  [작업A] 시작
  [작업A] 완료 (2초)
  [작업B] 시작
  [작업B] 완료 (2초)
  [작업C] 시작
  [작업C] 완료 (2초)

총 소요 시간: 6.0초


In [4]:
# 비동기 방식: 동시 실행
async def async_task(name, seconds):
    print(f"  [{name}] 시작")
    await asyncio.sleep(seconds)
    print(f"  [{name}] 완료 ({seconds}초)")

async def run_async():
    print("===== 비동기 실행 =====")
    start = time.time()
    await asyncio.gather(
        async_task("작업A", 2),
        async_task("작업B", 2),
        async_task("작업C", 2),
    )
    print(f"\n총 소요 시간: {time.time() - start:.1f}초")

await run_async()

===== 비동기 실행 =====
  [작업A] 시작
  [작업B] 시작
  [작업C] 시작
  [작업A] 완료 (2초)
  [작업B] 완료 (2초)
  [작업C] 완료 (2초)

총 소요 시간: 2.0초


결과를 비교합니다:

```
동기: 2초 + 2초 + 2초 = 6초 (순차 실행)
비동기: max(2초, 2초, 2초) = 2초 (동시 실행)
```


> `await`를 만나면, 이벤트 루프는 대기를 등록하고 바로 다음 작업을 시작합니다.  
> 세 작업의 "시작"이 거의 동시에 출력된 것이 그 증거입니다.

위에서 논의한 내용을 짧게 정리해보겠습니다  

![image.png](images/nb/nb_04_885e69e6.png)

### ✅ 체크포인트

1. `time.sleep(3)`과 `await asyncio.sleep(3)`의 핵심 차이는 무엇입니까?
2. 모델 추론처럼 CPU를 계속 사용하는 작업에서 `async/await`만으로 동시 처리가 안 되는 이유는 무엇입니까?

## 3. 문제 시연: 동기 추론이 서버를 멈추는 순간

---

> **학습 목표**
> - 동기 추론이 서버에 미치는 영향을 실제 코드로 재현합니다.
> - 동시 요청 시 응답 시간이 어떻게 누적되는지 측정합니다.
> - `def`와 `async def`의 동작 차이를 실험으로 확인합니다.

### 3.1 실험 설계

문제를 명확하게 보여주기 위해, 의도적으로 **추론 시간이 긴 서버**를 만들겠습니다.

실제 모델 추론 대신 `time.sleep()`으로 추론 지연을 시뮬레이션합니다.
이렇게 하면 추론 시간을 정확히 통제할 수 있어 실험 결과가 깔끔합니다.

In [6]:
%%writefile app/main_sync_problem.py
"""
Day 3 - 섹션 3: 동기 추론의 문제점을 보여주는 서버
두 가지 버전의 엔드포인트를 비교합니다.
"""
import time
from fastapi import FastAPI

app = FastAPI(title="Sync vs Async Problem Demo")

INFERENCE_TIME = 3   # 추론에 3초 걸린다고 가정

# ===== 버전 1: async def 안에서 동기 작업 (문제 있음) =====
@app.post("/predict/blocking")
async def predict_blocking():
    """
    ⚠️ 문제 버전: async def 안에서 time.sleep (동기 블로킹)
    이벤트 루프가 멈추므로, 동시 요청을 처리할 수 없습니다.
    """
    time.sleep(INFERENCE_TIME)   # 동기 블로킹 — 이벤트 루프가 멈춤
    return {"result": "완료", "method": "blocking", "duration": INFERENCE_TIME}


# ===== 버전 2: 일반 def (FastAPI가 스레드풀에서 실행) =====
@app.post("/predict/threadpool")
def predict_threadpool():
    """
    일반 def: FastAPI가 자동으로 별도 스레드에서 실행합니다.
    이벤트 루프는 블로킹되지 않지만, 스레드풀 크기에 제한이 있습니다.
    """
    time.sleep(INFERENCE_TIME)
    return {"result": "완료", "method": "threadpool", "duration": INFERENCE_TIME}


# 헬스체크: 서버가 응답 가능한 상태인지 확인용
@app.get("/health")
async def health():
    return {"status": "healthy"}

Overwriting app/main_sync_problem.py


In [7]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
serve_in_thread("app.main_sync_problem:app", port=8000)

서버 실행됨: http://127.0.0.1:8000


---

### 3.2 실험 1: 동시 요청 시 blocking 엔드포인트의 문제

3개의 요청을 **동시에** 보내고, 각각의 응답 시간을 측정합니다.

In [8]:
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def send_request(url, request_id):
    """단일 요청을 보내고 소요 시간을 측정합니다."""
    start = time.time()
    response = requests.post(url)
    elapsed = time.time() - start
    return {
        "request_id": request_id,
        "elapsed": round(elapsed, 1),
        "status": response.status_code,
    }

def concurrent_test(url, n_requests=3):
    """n개의 요청을 동시에 보냅니다."""
    print(f"\n{'='*55}")
    print(f"  {n_requests}개 동시 요청 → {url}")
    print(f"{'='*55}")

    start = time.time()

    with ThreadPoolExecutor(max_workers=n_requests) as executor:
        futures = {
            executor.submit(send_request, url, i+1): i
            for i in range(n_requests)
        }

        results = []
        for future in as_completed(futures):
            results.append(future.result())

    total = time.time() - start

    # 결과 출력 (요청 ID순으로 정렬)
    for r in sorted(results, key=lambda x: x["request_id"]):
        print(f"  요청 #{r['request_id']}: {r['elapsed']}초")

    print(f"\n  전체 소요 시간: {round(total, 1)}초")
    return total

In [9]:
# 실험 1: blocking 엔드포인트 (async def + time.sleep)
total_blocking = concurrent_test("http://localhost:8000/predict/blocking", n_requests=3)


  3개 동시 요청 → http://localhost:8000/predict/blocking
  요청 #1: 6.0초
  요청 #2: 3.0초
  요청 #3: 9.0초

  전체 소요 시간: 9.0초


결과를 분석합니다:

```
추론 시간이 3초인데, 3개 동시 요청의 결과:

요청 #1: 3초  (즉시 처리)
요청 #2: 6초  (요청 #1이 끝날 때까지 3초 대기 + 추론 3초)
요청 #3: 9초  (요청 #1, #2가 끝날 때까지 6초 대기 + 추론 3초)

→ 완전한 순차 실행입니다. 동시 처리가 전혀 이루어지지 않았습니다.
```

> 어떤 요청이 먼저 처리되는지는 실행마다 달라질 수 있어, 요청 번호와 시간의 짝은 위 표와 다를 수 있습니다.
> 핵심은 세 요청이 3초/6초/9초로 **하나씩 차례로** 끝난다는 점입니다.

---

### 3.3 실험 2: threadpool 엔드포인트와 비교

In [10]:
# 실험 2: threadpool 엔드포인트 (일반 def)
total_threadpool = concurrent_test("http://localhost:8000/predict/threadpool", n_requests=3)


  3개 동시 요청 → http://localhost:8000/predict/threadpool
  요청 #1: 3.0초
  요청 #2: 3.0초
  요청 #3: 3.0초

  전체 소요 시간: 3.0초


결과를 비교합니다:

![image.png](images/nb/nb_05_634541e7.png)

---

### 3.4 실험 3: blocking이 헬스체크까지 막는 현상

blocking 엔드포인트의 더 심각한 문제를 확인합니다.
추론 중에 헬스체크 요청이 들어오면 어떻게 될까요?

In [11]:
import threading

def test_health_during_inference(predict_url):
    """추론 중에 헬스체크가 응답하는지 테스트합니다."""

    results = {}

    def send_predict():
        start = time.time()
        requests.post(predict_url)
        results["predict"] = round(time.time() - start, 1)

    def send_health():
        time.sleep(0.5)   # 추론이 시작된 후 0.5초 뒤에 헬스체크
        start = time.time()
        resp = requests.get("http://localhost:8000/health")
        results["health"] = round(time.time() - start, 1)

    t1 = threading.Thread(target=send_predict)
    t2 = threading.Thread(target=send_health)
    t1.start()
    t2.start()
    t1.join()
    t2.join()

    return results

In [12]:
# blocking 버전
print("===== /predict/blocking 중 헬스체크 =====")
r = test_health_during_inference("http://localhost:8000/predict/blocking")
print(f"  추론 응답: {r['predict']}초")
print(f"  헬스체크 응답: {r['health']}초    ← 단순 상태 확인인데 2.5초 대기!")

print()

# threadpool 버전
print("===== /predict/threadpool 중 헬스체크 =====")
r = test_health_during_inference("http://localhost:8000/predict/threadpool")
print(f"  추론 응답: {r['predict']}초")
print(f"  헬스체크 응답: {r['health']}초    ← 즉시 응답!")

===== /predict/blocking 중 헬스체크 =====
  추론 응답: 3.0초
  헬스체크 응답: 2.5초    ← 단순 상태 확인인데 2.5초 대기!

===== /predict/threadpool 중 헬스체크 =====
  추론 응답: 3.0초
  헬스체크 응답: 0.0초    ← 즉시 응답!


> ⚠️ **blocking 버전에서는 헬스체크조차 대기합니다.**
>
> 이것이 실무에서 심각한 이유:
> - 로드밸런서는 헬스체크로 서버 상태를 판단합니다.
> - 헬스체크가 응답하지 않으면 "서버 다운"으로 판단하여 트래픽을 차단합니다.
> - 실제로는 서버가 살아있는데, 추론 때문에 응답을 못 하는 것뿐입니다.
> - 결과적으로 정상 서버가 풀에서 빠지는 심각한 문제가 발생합니다.

---

### 3.5 문제 원인 정리

```
[async def + 동기 작업의 문제]

async def predict_blocking():
    time.sleep(3)    ← 이벤트 루프 스레드에서 직접 실행
                        이벤트 루프가 3초간 완전히 멈춤
                        그 동안 다른 요청 처리 불가
                        헬스체크도 불가

[일반 def의 동작]

def predict_threadpool():
    time.sleep(3)    ← FastAPI가 별도 스레드풀(ThreadPoolExecutor)에서 실행
                        이벤트 루프는 자유로움
                        다른 요청 처리 가능
                        헬스체크 즉시 응답 가능
```

> 💡 **그러면 일반 def만 쓰면 되지 않습니까?**
>
> 일반 `def`를 사용하면 FastAPI가 기본 스레드풀에서 실행해줍니다.
> 이것만으로도 대부분의 경우 충분히 동작합니다.
>
> 하지만 두 가지 한계가 있습니다:
> 1. 기본 스레드풀 크기가 제한적입니다 (보통 40개).
> 2. 스레드풀 크기, 실행 방식을 직접 제어할 수 없습니다.
>
> 다음 섹션에서 배울 `run_in_executor` 패턴을 사용하면,
> 스레드풀을 명시적으로 제어하면서도 `async def`의 장점을 모두 활용할 수 있습니다.

---

### ✅ 체크포인트

1. `async def` 안에서 `time.sleep(3)`을 호출하면 왜 다른 요청까지 지연됩니까?
2. 일반 `def`로 선언된 엔드포인트는 FastAPI가 내부적으로 어떻게 처리합니까?
3. blocking 추론 중 헬스체크까지 막히면 실무에서 어떤 문제가 발생할 수 있습니까?

---

> **다음 섹션에서는** `run_in_executor` 패턴으로 이 문제를 해결합니다.
> `async def`의 장점을 유지하면서, 동기 작업도 이벤트 루프를 막지 않도록 만듭니다.

## 4. 해결 패턴: run_in_executor로 블로킹 방지하기

---

> **학습 목표**
> - `run_in_executor`의 동작 원리를 이해합니다.
> - 모델 추론을 이벤트 루프를 막지 않으면서 실행하는 패턴을 익힙니다.
> - ThreadPoolExecutor의 크기를 조절하는 방법을 알 수 있습니다.
> - Day 2에서 만든 API에 이 패턴을 적용할 수 있습니다.

### 4.1 run_in_executor란 무엇입니까?

`run_in_executor`는 **동기 함수를 별도 스레드(또는 프로세스)에서 실행**하고,
그 결과를 비동기적으로 기다리는 패턴입니다.

![image.png](images/nb/nb_06_44c87cc3.png)



코드로 보면 단 한 줄입니다:

```python
import asyncio

# Before: 이벤트 루프를 막는 방식
async def predict_blocking():
    result = model(input_tensor)     # 이벤트 루프 멈춤
    return result

# After: 이벤트 루프를 막지 않는 방식
async def predict_non_blocking():
    loop = asyncio.get_event_loop()
    result = await loop.run_in_executor(
        None,                        # None = 기본 스레드풀 사용
        model,                       # 실행할 함수
        input_tensor,                # 함수에 전달할 인자
    )
    return result
```

> `await loop.run_in_executor(None, func, arg)`
>
> 이 한 줄이 하는 일:
> 1. `func(arg)`를 별도 스레드에서 실행합니다.
> 2. 이벤트 루프는 그 동안 다른 작업을 처리합니다.
> 3. `func`이 완료되면 결과를 반환합니다.

### 4.2 실습: 세 가지 버전 비교

섹션 3의 실험에 `run_in_executor` 버전을 추가하여 세 가지를 비교합니다.

In [13]:
%%writefile app/main_async_solution.py
"""
Day 3 - 섹션 4: 세 가지 동시 처리 방식 비교
"""
import time
import asyncio
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI

app = FastAPI(title="Async Solution Demo")

INFERENCE_TIME = 3

# 커스텀 스레드풀 생성 (최대 4개 스레드)
inference_executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="inference")


def heavy_inference():
    """동기 함수: 모델 추론을 시뮬레이션합니다."""
    time.sleep(INFERENCE_TIME)
    return {"result": "완료", "duration": INFERENCE_TIME}


# ===== 버전 1: async def + 동기 작업 (문제 있음) =====
@app.post("/predict/v1-blocking")
async def predict_v1():
    """❌ 이벤트 루프를 막습니다."""
    time.sleep(INFERENCE_TIME)
    return {"method": "v1-blocking", "duration": INFERENCE_TIME}


# ===== 버전 2: 일반 def (FastAPI 자동 스레드풀) =====
@app.post("/predict/v2-def")
def predict_v2():
    """⭕ FastAPI가 자동으로 별도 스레드에서 실행합니다."""
    time.sleep(INFERENCE_TIME)
    return {"method": "v2-def", "duration": INFERENCE_TIME}


# ===== 버전 3: async def + run_in_executor (권장) =====
@app.post("/predict/v3-executor")
async def predict_v3():
    """✅ 명시적으로 스레드풀에 위임합니다."""
    loop = asyncio.get_event_loop()
    result = await loop.run_in_executor(
        inference_executor,    # 커스텀 스레드풀 사용
        heavy_inference,       # 실행할 동기 함수
    )
    return {"method": "v3-executor", **result}


@app.get("/health")
async def health():
    return {"status": "healthy"}

Writing app/main_async_solution.py


In [14]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
serve_in_thread("app.main_async_solution:app", port=8000)

서버 실행됨: http://127.0.0.1:8000


In [15]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

def concurrent_test(url, n_requests=3):
    """n개의 요청을 동시에 보내고 결과를 측정합니다."""
    def send(i):
        start = time.time()
        resp = requests.post(url)
        return {"id": i+1, "elapsed": round(time.time() - start, 1)}

    start = time.time()
    with ThreadPoolExecutor(max_workers=n_requests) as ex:
        futures = [ex.submit(send, i) for i in range(n_requests)]
        results = [f.result() for f in as_completed(futures)]
    total = round(time.time() - start, 1)

    for r in sorted(results, key=lambda x: x["id"]):
        print(f"  요청 #{r['id']}: {r['elapsed']}초")
    print(f"  전체: {total}초\n")
    return total

In [16]:
print("=" * 50)
print("버전 1: async def + time.sleep (blocking)")
print("=" * 50)
t1 = concurrent_test("http://localhost:8000/predict/v1-blocking")

버전 1: async def + time.sleep (blocking)
  요청 #1: 9.0초
  요청 #2: 3.0초
  요청 #3: 6.0초
  전체: 9.0초



In [17]:
print("=" * 50)
print("버전 2: 일반 def (FastAPI 자동 스레드풀)")
print("=" * 50)
t2 = concurrent_test("http://localhost:8000/predict/v2-def")

버전 2: 일반 def (FastAPI 자동 스레드풀)
  요청 #1: 3.0초
  요청 #2: 3.0초
  요청 #3: 3.0초
  전체: 3.0초



In [18]:
print("=" * 50)
print("버전 3: async def + run_in_executor (권장)")
print("=" * 50)
t3 = concurrent_test("http://localhost:8000/predict/v3-executor")

버전 3: async def + run_in_executor (권장)
  요청 #1: 3.0초
  요청 #2: 3.0초
  요청 #3: 3.0초
  전체: 3.0초



### 4.3 버전 2와 버전 3의 차이

결과만 보면 동일하지만, `run_in_executor`를 쓰는 핵심 이유가 있습니다:

```
일반 def (버전 2):
  → FastAPI 기본 스레드풀을 사용합니다.
  → 스레드풀 크기를 제어할 수 없습니다.
  → 다른 엔드포인트와 스레드풀을 공유합니다.

async def + run_in_executor (버전 3):
  → 추론 전용 스레드풀을 직접 만들어 사용합니다.
  → 크기를 제어할 수 있습니다 (max_workers=4).
  → 추론 부하가 다른 엔드포인트에 영향을 주지 않습니다.
```

> 간단한 프로젝트에서는 일반 `def`로 충분하지만,
> 추론 전용 스레드풀을 분리하고 싶다면 `run_in_executor`를 사용합니다.
> 이 과정에서는 `run_in_executor` 패턴을 기본으로 사용합니다.

### 4.4 Day 2 API에 적용하기

Day 2에서 만든 실제 추론 API(`app/main.py`)에 `run_in_executor`를 적용합니다.

In [19]:
%%writefile app/main_v2.py
"""
Day 3 - Day 2 API에 비동기 패턴 적용
app/main.py의 개선 버전입니다.
"""
import io
import base64
import asyncio
from concurrent.futures import ThreadPoolExecutor

import torch
import numpy as np
from PIL import Image
from fastapi import FastAPI, HTTPException

from app.schemas import (
    PixelPredictRequest,
    ImagePredictRequest,
    PredictResponse,
)
from app.model_utils import load_model, predict, preprocess


# ===== 앱 생성 =====
app = FastAPI(
    title="MNIST Prediction API (Async)",
    description="비동기 처리가 적용된 MNIST 추론 API",
    version="2.0.0",
)

# ===== 추론 전용 스레드풀 =====
inference_executor = ThreadPoolExecutor(
    max_workers=4,
    thread_name_prefix="inference",
)

# ===== 모델 로드 =====
MODEL_PATH = "models/mnist_state_dict.pth"
model = load_model(MODEL_PATH)


# ===== 동기 추론 함수 (스레드풀에서 실행될 함수) =====
def run_inference(image_tensor: torch.Tensor) -> dict:
    """모델 추론을 수행합니다. 이 함수는 별도 스레드에서 실행됩니다."""
    return predict(model, image_tensor)


# ===== 엔드포인트 =====

@app.get("/health", tags=["System"])
async def health_check():
    return {"status": "healthy", "model_loaded": model is not None}


@app.post("/predict/pixels", response_model=PredictResponse, tags=["Inference"])
async def predict_from_pixels(request: PixelPredictRequest):
    """비동기 버전: 픽셀 배열로 추론"""
    try:
        # 전처리 (가벼운 작업 — 이벤트 루프에서 직접 수행)
        pixel_array = np.array(request.pixels, dtype=np.float32)
        pixel_tensor = torch.from_numpy(pixel_array)
        pixel_tensor = (pixel_tensor - 0.1307) / 0.3081
        pixel_tensor = pixel_tensor.unsqueeze(0).unsqueeze(0)

        # 추론 (무거운 작업 — 별도 스레드에서 실행)
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(
            inference_executor,
            run_inference,
            pixel_tensor,
        )

        return PredictResponse(
            success=True,
            predicted_class=result["predicted_class"],
            confidence=result["confidence"],
            probabilities=result["probabilities"] if request.return_probabilities else None,
        )

    except Exception as e:
        raise HTTPException(status_code=400, detail=f"추론 실패: {str(e)}")


@app.post("/predict/image", response_model=PredictResponse, tags=["Inference"])
async def predict_from_image(request: ImagePredictRequest):
    """비동기 버전: Base64 이미지로 추론"""
    try:
        # 전처리 (가벼운 작업)
        image_bytes = base64.b64decode(request.image_base64)
        image = Image.open(io.BytesIO(image_bytes))
        image_tensor = preprocess(image).unsqueeze(0)

        # 추론 (무거운 작업 — 별도 스레드에서 실행)
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(
            inference_executor,
            run_inference,
            image_tensor,
        )

        return PredictResponse(
            success=True,
            predicted_class=result["predicted_class"],
            confidence=result["confidence"],
            probabilities=result["probabilities"] if request.return_probabilities else None,
        )

    except base64.binascii.Error:
        raise HTTPException(status_code=400, detail="유효하지 않은 Base64 문자열입니다.")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"이미지 처리 실패: {str(e)}")

Writing app/main_v2.py


변경된 부분을 정리합니다:

```
Day 2 (app/main.py)                    Day 3 (app/main_v2.py)
──────────────────                     ──────────────────────

def predict_from_pixels(...):          async def predict_from_pixels(...):
    ...                                    ...
    result = predict(model, tensor)        result = await loop.run_in_executor(
    ...                                        inference_executor,
                                               run_inference,
                                               tensor,
                                           )
                                           ...

변경: 2줄                               추가: inference_executor, run_inference
```

> 핵심 변경은 **추론 호출 한 줄**뿐입니다.
> 나머지 전처리, 검증, 에러 처리 코드는 그대로 유지됩니다.

### 4.5 스레드풀 크기 가이드

```
CPU 추론 (GPU 없음):
  - CPU 코어 수와 비슷하게 설정합니다.
  - 예: 4코어 → max_workers=4
  - 너무 많으면 컨텍스트 스위칭 오버헤드가 발생합니다.

GPU 추론:
  - GPU 메모리 제한이 있으므로 1~2로 설정합니다.
  - GPU는 내부적으로 병렬 처리하므로, 스레드를 늘려도 속도 향상이 크지 않습니다.
```

In [20]:
import os

# CPU 코어 수 확인
cpu_count = os.cpu_count()
print(f"CPU 코어 수: {cpu_count}")
print(f"권장 max_workers (CPU 추론): {cpu_count}")
print(f"권장 max_workers (GPU 추론): 1~2")

CPU 코어 수: 32
권장 max_workers (CPU 추론): 32
권장 max_workers (GPU 추론): 1~2


### 4.6 정리: 어떤 패턴을 선택해야 합니까?


![image.png](images/nb/nb_07_391cb067.png)


> 이 과정의 프로젝트에서는 `run_in_executor` 패턴을 사용합니다.
> 한번 익혀두면 실무에서도 그대로 적용할 수 있기 때문입니다.

---

### ✅ 체크포인트

1. `run_in_executor`가 이벤트 루프 블로킹을 방지하는 원리는 무엇입니까?
2. `run_in_executor`의 첫 번째 인자에 `None`을 넣으면 어떤 스레드풀이 사용됩니까?
3. 일반 `def`와 `async def + run_in_executor`의 핵심 차이는 무엇입니까?
4. GPU 추론 시 스레드풀 크기를 1~2로 제한하는 이유는 무엇입니까?

---

> **다음 섹션에서는** 서버의 안정성을 높이기 위한 에러 핸들링과 로깅을 다룹니다.


## 5. 에러 핸들링과 로깅

---

> **학습 목표**
> - 예외 발생 시 서버가 죽지 않고, 안전한 에러 응답을 반환하는 구조를 만들 수 있습니다.
> - Python logging 모듈을 사용하여 로그를 남길 수 있습니다.

### 5.1 왜 에러 핸들링이 중요합니까?

입력이 올바르더라도 **서버 내부에서 에러가 발생**할 수 있습니다.

```
- 모델 파일이 손상되어 추론 중 에러 발생
- GPU 메모리 부족 (OOM)
- 이미지 디코딩 실패 (손상된 파일)
```

에러를 처리하지 않으면:

```
→ 스택 트레이스가 클라이언트에 노출 (보안 위험)
→ 에러 원인 추적 불가 (로그가 없음)

에러 핸들링 적용 후:
→ 서버 로그에 상세 정보 기록
→ 클라이언트에는 안전한 메시지만 반환
→ 서버는 계속 동작
```

### 5.2 글로벌 Exception Handler

매 엔드포인트마다 try/except를 반복하는 대신,
**글로벌 핸들러 하나**로 모든 예외를 중앙에서 처리합니다.

In [21]:
%%writefile app/error_handlers.py
"""
Day 3 - 글로벌 에러 핸들러
"""
import traceback
import logging

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

logger = logging.getLogger("ml_api")


def register_error_handlers(app: FastAPI):
    """FastAPI 앱에 글로벌 에러 핸들러를 등록합니다."""

    @app.exception_handler(Exception)
    async def general_error_handler(request: Request, exc: Exception):
        """모든 예외를 잡아서 안전한 응답을 반환합니다."""
        logger.error(
            f"에러 발생: {type(exc).__name__}: {exc}\n"
            f"경로: {request.method} {request.url}\n"
            f"스택 트레이스:\n{traceback.format_exc()}"
        )
        return JSONResponse(
            status_code=500,
            content={
                "success": False,
                "error": "서버 내부 오류가 발생했습니다.",
            }
            # ⚠️ 클라이언트에게는 상세 정보를 노출하지 않습니다.
            # 상세 정보는 서버 로그에만 기록됩니다.
        )

Writing app/error_handlers.py


> 핵심 원칙:
> - **로그에는** 스택 트레이스, 요청 경로, 원본 에러 등 상세 정보를 기록합니다.
> - **클라이언트에게는** "서버 내부 오류가 발생했습니다"만 반환합니다.
> - 내부 코드 경로, 라이브러리 버전 등이 노출되면 보안 위험이 됩니다.

> 💡 **더 알고 싶다면**
>
> 실무에서는 커스텀 예외 클래스(`InferenceError`, `PreprocessError` 등)를 정의하여
> 에러 유형별로 다른 상태 코드를 반환하기도 합니다.
> 이 과정에서는 범용 핸들러 하나로 충분합니다.

### 5.3 Python logging 설정

`print()` 대신 `logging` 모듈을 사용해야 하는 이유:

```
print("에러 발생")           → 시간 정보 없음, 심각도 구분 없음, 파일 저장 불가

logger.error("에러 발생")    → 2024-01-15 14:30:22 ERROR [ml_api] 에러 발생
                               시간, 심각도, 모듈명이 자동 포함
```

In [22]:
%%writefile app/logger_config.py
"""
Day 3 - 로깅 설정
"""
import logging
import sys


def setup_logger(name: str = "ml_api", level: str = "INFO") -> logging.Logger:
    """콘솔 로거를 설정합니다."""
    logger = logging.getLogger(name)
    logger.setLevel(getattr(logging, level))

    if logger.handlers:
        return logger

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.DEBUG)

    formatter = logging.Formatter(
        fmt="%(asctime)s %(levelname)-8s [%(name)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    return logger

Writing app/logger_config.py


> 실무에서는 파일 핸들러나 외부 로그 서비스(CloudWatch, Datadog 등)로
> 로그를 보내지만, 이 과정에서는 콘솔 로그로 충분합니다.

In [23]:
# 로거 테스트
from app.logger_config import setup_logger

logger = setup_logger("ml_api")

logger.info("서버가 시작되었습니다.")
logger.warning("GPU 메모리가 80%를 초과했습니다.")
logger.error("모델 추론 중 에러가 발생했습니다.")

2026-08-14 16:49:36 INFO     [ml_api] 서버가 시작되었습니다.
2026-08-14 16:49:36 WARNING  [ml_api] GPU 메모리가 80%를 초과했습니다.
2026-08-14 16:49:36 ERROR    [ml_api] 모델 추론 중 에러가 발생했습니다.


```
2024-01-15 14:30:22 INFO     [ml_api] 서버가 시작되었습니다.
2024-01-15 14:30:22 WARNING  [ml_api] GPU 메모리가 80%를 초과했습니다.
2024-01-15 14:30:22 ERROR    [ml_api] 모델 추론 중 에러가 발생했습니다.
```

로그 레벨의 의미:

```
INFO     → 정상 동작 기록 (서버 시작, 모델 로드 등)
WARNING  → 주의가 필요한 상황 (메모리 부족 임박 등)
ERROR    → 에러 발생 (추론 실패 등)
CRITICAL → 심각한 에러 (서버 다운 위험)
```

### 5.4 요청/응답 로깅 미들웨어

모든 요청과 응답을 자동으로 로깅하는 미들웨어입니다.
아래 코드를 그대로 사용하시면 됩니다.

In [24]:
%%writefile app/middleware.py
"""
Day 3 - 요청/응답 로깅 미들웨어
모든 요청의 메서드, 경로, 응답 시간, 상태 코드를 자동 로깅합니다.
"""
import time
import logging
from fastapi import Request
from starlette.middleware.base import BaseHTTPMiddleware

logger = logging.getLogger("ml_api")


class RequestLoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        start_time = time.time()
        response = await call_next(request)
        duration = round(time.time() - start_time, 3)

        log_message = (
            f"{request.method} {request.url.path} "
            f"→ {response.status_code} "
            f"({duration}s)"
        )

        if response.status_code >= 500:
            logger.error(log_message)
        elif response.status_code >= 400:
            logger.warning(log_message)
        else:
            logger.info(log_message)

        response.headers["X-Process-Time"] = str(duration)
        return response

Writing app/middleware.py


이 미들웨어를 등록하면 서버 로그가 이렇게 보입니다:

```
2024-01-15 14:30:22 INFO     [ml_api] GET /health → 200 (0.001s)
2024-01-15 14:30:23 INFO     [ml_api] POST /predict/pixels → 200 (0.045s)
2024-01-15 14:30:24 WARNING  [ml_api] POST /predict/pixels → 422 (0.002s)
```

> 미들웨어는 `app.add_middleware(RequestLoggingMiddleware)`로 등록합니다.
> 섹션 6의 최종 서버 코드에서 실제로 적용합니다.

---

### ✅ 체크포인트

1. 글로벌 Exception Handler를 사용하면 어떤 반복을 줄일 수 있습니까?
2. 클라이언트에게 스택 트레이스를 노출하면 안 되는 이유는 무엇입니까?
3. `logging` 모듈이 `print()`보다 나은 점은 무엇입니까?

---

> **다음 섹션에서는** 오늘 배운 모든 내용을 종합하여,
> 동시 요청 테스트로 개선 효과를 직접 측정합니다.


## 6. 실습: 최종 서버 + 동시 요청 테스트

---

> **실습 목표**
> - 비동기 + 에러 핸들링 + 로깅이 적용된 최종 서버를 실행합니다.
> - 동시 요청 테스트로 동작을 확인합니다.
> - 에러 핸들링이 정상 동작하는지 확인합니다.

### 6.0 사전 준비: 의존 파일 확인  


이 섹션은 Day 1~3에서 생성한 파일들을 사용합니다.
아래 셀을 실행하면 필요한 파일이 있는지 확인하고, 없으면 자동으로 생성합니다.

> ⚠️ Day 1~3을 순서대로 진행한 분은 이 셀을 건너뛰어도 됩니다.
> 이 섹션부터 바로 시작하는 분만 실행하세요.

In [25]:


import os

# 필요한 폴더 생성
for d in ["app", "models", "data"]:
    os.makedirs(d, exist_ok=True)

# 1. app/model_utils.py (Day 1)
if True:
    print("⚠️ app/model_utils.py 없음 → 생성합니다.")
    with open("app/model_utils.py", "w", encoding="utf-8") as f:
        f.write('''import torch
import torch.nn as nn
from torchvision import transforms

class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64*7*7, 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

preprocess = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

CLASS_NAMES = [str(i) for i in range(10)]

def load_model(model_path, num_classes=10):
    model = SimpleClassifier(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
    model.eval()
    return model

def predict(model, image_tensor):
    with torch.no_grad():
        output = model(image_tensor)
        probs = torch.softmax(output, dim=1)[0]
        idx = probs.argmax().item()
        return {
            "predicted_class": CLASS_NAMES[idx],
            "confidence": round(probs[idx].item(), 4),
            "probabilities": {CLASS_NAMES[i]: round(probs[i].item(), 4) for i in range(len(CLASS_NAMES))},
        }
''')
    print("  ✅ app/model_utils.py 생성 완료")
else:
    print("✅ app/model_utils.py 있음")

# 2. app/schemas.py (Day 2)
if True:
    print("⚠️ app/schemas.py 없음 → 생성합니다.")
    with open("app/schemas.py", "w", encoding="utf-8") as f:
        f.write('''from pydantic import BaseModel, Field, field_validator
from typing import Optional

class PixelPredictRequest(BaseModel):
    pixels: list[list[float]] = Field(..., description="28x28 픽셀 배열")
    return_probabilities: bool = Field(default=False)
    @field_validator("pixels")
    @classmethod
    def validate_pixels(cls, v):
        if len(v) != 28:
            raise ValueError(f"28행이어야 합니다. 현재: {len(v)}행")
        for i, row in enumerate(v):
            if len(row) != 28:
                raise ValueError(f"각 행은 28열이어야 합니다. {i}번째 행: {len(row)}열")
        return v

class ImagePredictRequest(BaseModel):
    image_base64: str = Field(..., min_length=1)
    return_probabilities: bool = Field(default=False)

class PredictResponse(BaseModel):
    success: bool = Field(description="성공 여부")
    predicted_class: str = Field(description="예측 숫자 (0~9)")
    confidence: float = Field(description="확신도", ge=0.0, le=1.0)
    probabilities: Optional[dict[str, float]] = Field(default=None)
''')
    print("  ✅ app/schemas.py 생성 완료")
else:
    print("✅ app/schemas.py 있음")

# 3. app/error_handlers.py (Day 3 섹션 5)
if True:
    print("⚠️ app/error_handlers.py 없음 → 생성합니다.")
    with open("app/error_handlers.py", "w", encoding="utf-8") as f:
        f.write('''import traceback, logging
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

logger = logging.getLogger("ml_api")

def register_error_handlers(app: FastAPI):
    @app.exception_handler(Exception)
    async def general_error_handler(request: Request, exc: Exception):
        logger.error(f"에러: {type(exc).__name__}: {exc}\\n경로: {request.method} {request.url}\\n{traceback.format_exc()}")
        return JSONResponse(status_code=500, content={"success": False, "error": "서버 내부 오류가 발생했습니다."})
''')
    print("  ✅ app/error_handlers.py 생성 완료")
else:
    print("✅ app/error_handlers.py 있음")

# 4. app/logger_config.py (Day 3 섹션 5)
if True:
    print("⚠️ app/logger_config.py 없음 → 생성합니다.")
    with open("app/logger_config.py", "w", encoding="utf-8") as f:
        f.write('''import logging, sys

def setup_logger(name="ml_api", level="INFO"):
    logger = logging.getLogger(name)
    logger.setLevel(getattr(logging, level))
    if logger.handlers:
        return logger
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)-8s [%(name)s] %(message)s", "%Y-%m-%d %H:%M:%S"))
    logger.addHandler(handler)
    return logger
''')
    print("  ✅ app/logger_config.py 생성 완료")
else:
    print("✅ app/logger_config.py 있음")

# 5. app/middleware.py (Day 3 섹션 5)
if True:
    print("⚠️ app/middleware.py 없음 → 생성합니다.")
    with open("app/middleware.py", "w", encoding="utf-8") as f:
        f.write('''import time, logging
from fastapi import Request
from starlette.middleware.base import BaseHTTPMiddleware

logger = logging.getLogger("ml_api")

class RequestLoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        start = time.time()
        response = await call_next(request)
        duration = round(time.time() - start, 3)
        msg = f"{request.method} {request.url.path} -> {response.status_code} ({duration}s)"
        if response.status_code >= 500: logger.error(msg)
        elif response.status_code >= 400: logger.warning(msg)
        else: logger.info(msg)
        response.headers["X-Process-Time"] = str(duration)
        return response
''')
    print("  ✅ app/middleware.py 생성 완료")
else:
    print("✅ app/middleware.py 있음")

# 6. 모델 파일 (Day 1)
if not os.path.exists("models/mnist_state_dict.pth"):
    print("⚠️ 모델 파일 없음 → MNIST 모델을 학습하여 생성합니다. (약 1~2분 소요)")
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader
    from app.model_utils import SimpleClassifier

    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    train_data = DataLoader(datasets.MNIST("data", train=True, download=True, transform=transform), batch_size=64, shuffle=True)

    model = SimpleClassifier(10)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(2):
        for batch_idx, (images, labels) in enumerate(train_data):
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            if (batch_idx+1) % 300 == 0:
                print(f"  Epoch {epoch+1}, Batch {batch_idx+1}/{len(train_data)}, Loss: {loss.item():.4f}")

    torch.save(model.state_dict(), "models/mnist_state_dict.pth")
    print("  ✅ models/mnist_state_dict.pth 생성 완료")
else:
    print("✅ models/mnist_state_dict.pth 있음")

print("\n🎉 모든 의존 파일이 준비되었습니다. 다음 셀로 진행하세요.")

⚠️ app/model_utils.py 없음 → 생성합니다.
  ✅ app/model_utils.py 생성 완료
⚠️ app/schemas.py 없음 → 생성합니다.
  ✅ app/schemas.py 생성 완료
⚠️ app/error_handlers.py 없음 → 생성합니다.
  ✅ app/error_handlers.py 생성 완료
⚠️ app/logger_config.py 없음 → 생성합니다.
  ✅ app/logger_config.py 생성 완료
⚠️ app/middleware.py 없음 → 생성합니다.
  ✅ app/middleware.py 생성 완료
✅ models/mnist_state_dict.pth 있음

🎉 모든 의존 파일이 준비되었습니다. 다음 셀로 진행하세요.


### 6.1 최종 서버 코드 통합

Day 2 API에 오늘 배운 비동기 처리, 에러 핸들링, 로깅을 모두 적용한 최종 버전입니다.

In [26]:
%%writefile app/main_final.py
"""
Day 3 최종 버전 - 비동기 + 에러 핸들링 + 로깅
"""
import io
import base64
import asyncio
from concurrent.futures import ThreadPoolExecutor

import torch
import numpy as np
from PIL import Image
from fastapi import FastAPI, HTTPException

from app.schemas import PixelPredictRequest, ImagePredictRequest, PredictResponse
from app.model_utils import load_model, predict, preprocess
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware


logger = setup_logger("ml_api")

app = FastAPI(
    title="MNIST Prediction API",
    description="비동기 처리, 에러 핸들링, 로깅이 적용된 MNIST 추론 API",
    version="3.0.0",
)

app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

inference_executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="inference")

MODEL_PATH = "models/mnist_state_dict.pth"
model = None


@app.on_event("startup")
async def startup():
    global model
    logger.info(f"모델 로드 중: {MODEL_PATH}")
    model = load_model(MODEL_PATH)
    logger.info("모델 로드 완료")


def run_inference(image_tensor: torch.Tensor) -> dict:
    """별도 스레드에서 실행되는 추론 함수"""
    if model is None:
        raise RuntimeError("모델이 로드되지 않았습니다")
    return predict(model, image_tensor)


@app.get("/health", tags=["System"])
async def health_check():
    return {
        "status": "healthy" if model is not None else "loading",
        "model_loaded": model is not None,
    }


@app.get("/model/info", tags=["System"])
async def model_info():
    from app.model_utils import CLASS_NAMES
    total_params = sum(p.numel() for p in model.parameters())
    return {
        "model_name": "SimpleClassifier",
        "model_path": MODEL_PATH,
        "num_classes": len(CLASS_NAMES),
        "classes": CLASS_NAMES,
        "total_parameters": total_params,
    }


@app.post("/predict/pixels", response_model=PredictResponse, tags=["Inference"])
async def predict_from_pixels(request: PixelPredictRequest):
    try:
        pixel_array = np.array(request.pixels, dtype=np.float32)
        pixel_tensor = torch.from_numpy(pixel_array)
        pixel_tensor = (pixel_tensor - 0.1307) / 0.3081
        pixel_tensor = pixel_tensor.unsqueeze(0).unsqueeze(0)
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"전처리 실패: {str(e)}")

    try:
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(inference_executor, run_inference, pixel_tensor)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"추론 실패: {str(e)}")

    return PredictResponse(
        success=True,
        predicted_class=result["predicted_class"],
        confidence=result["confidence"],
        probabilities=result["probabilities"] if request.return_probabilities else None,
    )


@app.post("/predict/image", response_model=PredictResponse, tags=["Inference"])
async def predict_from_image(request: ImagePredictRequest):
    try:
        image_bytes = base64.b64decode(request.image_base64)
        image = Image.open(io.BytesIO(image_bytes))
        image_tensor = preprocess(image).unsqueeze(0)
    except base64.binascii.Error:
        raise HTTPException(status_code=400, detail="유효하지 않은 Base64 문자열입니다.")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"이미지 처리 실패: {str(e)}")

    try:
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(inference_executor, run_inference, image_tensor)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"추론 실패: {str(e)}")

    return PredictResponse(
        success=True,
        predicted_class=result["predicted_class"],
        confidence=result["confidence"],
        probabilities=result["probabilities"] if request.return_probabilities else None,
    )

Writing app/main_final.py


> Day 2의 `app/main.py`와 비교했을 때 변경된 부분:
> - `async def` + `run_in_executor`로 추론 비동기화
> - `setup_logger`로 구조화된 로깅
> - `register_error_handlers`로 글로벌 에러 처리
> - `RequestLoggingMiddleware`로 모든 요청 자동 로깅

---

### 6.2 서버 실행 및 테스트

In [32]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
serve_in_thread("app.main_final:app", port=8000)

2026-08-14 17:08:56 INFO     [ml_api] 모델 로드 중: models/mnist_state_dict.pth
2026-08-14 17:08:56 INFO     [ml_api] 모델 로드 완료
서버 실행됨: http://127.0.0.1:8000


#### 동시 요청 테스트

섹션 3에서 사용한 `concurrent_test()` 함수를 재사용하여 최종 서버를 테스트합니다.

In [28]:
import requests
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from torchvision import datasets

test_dataset = datasets.MNIST(root="data", train=False, download=True)

def concurrent_pixel_test(n_requests=3):
    """실제 모델 추론으로 동시 요청을 테스트합니다."""
    def send(i):
        image, label = test_dataset[i % len(test_dataset)]
        pixels = (np.array(image) / 255.0).tolist()
        start = time.time()
        resp = requests.post(
            "http://localhost:8000/predict/pixels",
            json={"pixels": pixels},
            timeout=30,
        )
        return {
            "id": i + 1,
            "elapsed": round(time.time() - start, 2),
            "status": resp.status_code,
        }

    print(f"\n{'='*50}")
    print(f"  {n_requests}개 동시 요청 (실제 추론)")
    print(f"{'='*50}")

    start = time.time()
    with ThreadPoolExecutor(max_workers=n_requests) as ex:
        futures = [ex.submit(send, i) for i in range(n_requests)]
        results = [f.result() for f in as_completed(futures)]
    total = round(time.time() - start, 2)

    for r in sorted(results, key=lambda x: x["id"]):
        print(f"  요청 #{r['id']}: {r['elapsed']}초 (HTTP {r['status']})")
    print(f"  전체: {total}초")

In [29]:
# 동시 요청 수를 늘려가며 테스트
for n in [1, 2, 4, 8]:
    concurrent_pixel_test(n)
    time.sleep(1)


  1개 동시 요청 (실제 추론)
2026-08-14 16:50:29 INFO     [ml_api] POST /predict/pixels -> 200 (0.006s)
  요청 #1: 0.01초 (HTTP 200)
  전체: 0.02초

  2개 동시 요청 (실제 추론)
2026-08-14 16:50:30 INFO     [ml_api] POST /predict/pixels -> 200 (0.004s)
2026-08-14 16:50:30 INFO     [ml_api] POST /predict/pixels -> 200 (0.006s)
  요청 #1: 0.01초 (HTTP 200)
  요청 #2: 0.01초 (HTTP 200)
  전체: 0.01초

  4개 동시 요청 (실제 추론)
2026-08-14 16:50:31 INFO     [ml_api] POST /predict/pixels -> 200 (0.006s)
2026-08-14 16:50:31 INFO     [ml_api] POST /predict/pixels -> 200 (0.007s)
2026-08-14 16:50:31 INFO     [ml_api] POST /predict/pixels -> 200 (0.008s)
2026-08-14 16:50:31 INFO     [ml_api] POST /predict/pixels -> 200 (0.009s)
  요청 #1: 0.01초 (HTTP 200)
  요청 #2: 0.01초 (HTTP 200)
  요청 #3: 0.01초 (HTTP 200)
  요청 #4: 0.01초 (HTTP 200)
  전체: 0.02초

  8개 동시 요청 (실제 추론)
2026-08-14 16:50:32 INFO     [ml_api] POST /predict/pixels -> 200 (0.01s)
2026-08-14 16:50:32 INFO     [ml_api] POST /predict/pixels -> 200 (0.01s)
2026-08-14 16:50:32 INFO     

---

### 6.3 에러 핸들링 동작 확인

In [31]:
print("=" * 50)
print("  에러 핸들링 테스트")
print("=" * 50)

# 정상 요청
image, label = test_dataset[0]
pixels = (np.array(image) / 255.0).tolist()
resp = requests.post("http://localhost:8000/predict/pixels", json={"pixels": pixels})
print(f"\n[정상 요청] 상태: {resp.status_code}, 예측: {resp.json()['predicted_class']}")

# 잘못된 픽셀 크기
resp = requests.post(
    "http://localhost:8000/predict/pixels",
    json={"pixels": [[0.0] * 14 for _ in range(14)]}
)
print(f"[잘못된 크기] 상태: {resp.status_code}")

# 잘못된 Base64
resp = requests.post(
    "http://localhost:8000/predict/image",
    json={"image_base64": "not_valid!!!"}
)
print(f"[잘못된 Base64] 상태: {resp.status_code}, 에러: {resp.json().get('detail', 'N/A')}")

# 헬스체크
resp = requests.get("http://localhost:8000/health")
print(f"[헬스체크] 상태: {resp.status_code}, 응답: {resp.json()}")

  에러 핸들링 테스트
2026-08-14 16:50:38 INFO     [ml_api] POST /predict/pixels -> 200 (0.003s)

[정상 요청] 상태: 200, 예측: 7
2026-08-14 16:50:38 WARNING  [ml_api] POST /predict/pixels -> 422 (0.001s)
[잘못된 크기] 상태: 422
2026-08-14 16:50:38 WARNING  [ml_api] POST /predict/image -> 400 (0.001s)
[잘못된 Base64] 상태: 400, 에러: 이미지 처리 실패: cannot identify image file <_io.BytesIO object at 0x7366b8bf92b0>
2026-08-14 16:50:38 INFO     [ml_api] GET /health -> 200 (0.0s)
[헬스체크] 상태: 200, 응답: {'status': 'healthy', 'model_loaded': True}


2026-08-14 17:01:25 INFO     [ml_api] GET /openapi.json -> 200 (0.005s)
2026-08-14 17:01:25 INFO     [ml_api] GET /health -> 200 (0.001s)


> 모든 비정상 요청이 **서버를 죽이지 않고**, 적절한 상태 코드와 메시지를 반환합니다.

---

### 추가 실험 A ~ H (내가 더 해본 것)

바로 위 6.2절에서 동시 요청을 1, 2, 4, 8개로 늘려가며 재봤는데
0.02 / 0.01 / 0.02 / 0.03초로 거의 같았다.
요청을 8배로 늘렸는데 시간이 안 늘었으니 얼핏 비동기 처리가 잘 된 것처럼 읽힌다.
그런데 서버 로그를 보면 추론 한 건이 6밀리초다. 그렇게 가벼운 일이면
`run_in_executor` 를 붙이든 빼든 이 표에는 차이가 안 나타날 것 같았다.
그러면 이 숫자는 개선했다는 근거로 쓰기 어렵다는 생각이 들었다.

그래서 A부터 H까지 여덟 번 더 재봤다. 순서대로 읽으면 이렇게 이어진다.

```
A   부하를 300개까지 올려봤다               완전히 직렬로 나왔다
B   서버를 노트북 커널 밖으로 빼봤다          A 의 직렬은 측정 환경 때문이었다
C   run_in_executor 를 뺀 서버와 비교        차이를 못 봤다
D   /health 로 클라이언트 천장을 쟀다         서버가 병목이 된 적이 없었다
E   추론을 ollama 로 바꿔 네 방식 비교        헬스체크가 갈렸다
F   스레드를 몇 개 쓰는지 세봤다              측정에 결함이 있었다
F2  방식마다 서버를 새로 띄워 다시 셌다        표가 깨끗해졌다
G   재현이 안 되던 숫자를 3번씩 다시 쟀다      이상치였다
H   CPU 바운드로 돌아와 결론을 냈다           파이토치가 GIL 을 놓는다
```

여덟 번 중 네 번은 재려던 것이 아니라 **내 측정 도구가 문제**였다.
그게 이 실험들의 실제 내용에 가깝다.

각 코드 셀 맨 위 주석에 그때그때 무슨 생각으로 짰는지 적어뒀다.

#### A. 부하를 300개까지 올려봤다

8개로는 아무것도 안 보였으니 막히는 게 보일 때까지 올려보기로 했다.
300개쯤이면 보일 거라고 예상했다.

돌리기 전에 판독 기준을 먼저 정해뒀다. 숫자를 보고 나서 해석을 붙이면
끼워맞추기가 될 것 같아서다. 요청이 37.5배 늘었을 때 시간도 37.5배 가까이 늘면 완전 직렬,
거의 안 늘면 아직 병목이 아니라고 보기로 했다.

In [34]:
# %load ../DP03/build/실험A_부하테스트.py
# [내 실험 A] 8개로는 아무것도 안 보여서, 부하를 올려 다시 쟀다
#
# 6.2절 결과는 1개일 때 0.02초, 8개일 때 0.03초로 거의 같았다. 추론 한 건이 6밀리초라
# 요청을 8배로 늘려도 서버가 전혀 힘들어하지 않은 것이다.
# 그 표로는 run_in_executor 를 붙인 효과가 있는지 없는지 알 수가 없다.
# 그래서 막히는 게 보이기 시작할 때까지 부하를 올려보기로 했다. 300개쯤이면 보일 거라고 예상했다.

import time
import logging
import numpy as np
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed


def load_test(n_requests, url="http://localhost:8000/predict/pixels"):
    """요청을 n개 동시에 보내고, 개별 줄 대신 요약만 낸다."""
    # 300개면 서버 로그가 300줄 찍혀서 정작 결과를 못 본다. 재는 동안만 조용히 시킨다
    log = logging.getLogger("ml_api")
    prev_level = log.level
    log.setLevel(logging.WARNING)

    def send(i):
        image, _ = test_dataset[i % len(test_dataset)]      # 6.2절에서 만든 test_dataset 을 그대로 쓴다
        pixels = (np.array(image) / 255.0).tolist()          # 0~1 원본값. 정규화는 서버가 한다
        t0 = time.time()
        resp = requests.post(url, json={"pixels": pixels}, timeout=60)
        return time.time() - t0, resp.status_code

    try:
        start = time.time()
        with ThreadPoolExecutor(max_workers=n_requests) as ex:
            futures = [ex.submit(send, i) for i in range(n_requests)]
            results = [f.result() for f in as_completed(futures)]
        total = time.time() - start
    finally:
        log.setLevel(prev_level)      # 무슨 일이 있어도 로그 레벨은 되돌린다

    times = sorted(r[0] for r in results)
    codes = {}
    for _, c in results:
        codes[c] = codes.get(c, 0) + 1

    # 전체 시간만 보면 "300개를 2초에 처리했다"로 끝나는데,
    # 그중 한 명이 1.9초를 기다렸다면 그 사람 입장에서는 느린 서버다.
    # 평균은 웃는데 꼬리가 우는 경우를 잡으려고 95%와 최대를 같이 찍는다
    p50 = times[len(times) // 2]
    p95 = times[int(len(times) * 0.95) - 1]
    print(f"  {n_requests:>4}개 | 전체 {total:6.2f}초 | "
          f"개별 중앙 {p50*1000:6.1f}ms  95% {p95*1000:7.1f}ms  최대 {times[-1]*1000:7.1f}ms | {codes}")
    return total


print("=" * 78)
print("  부하를 올려가며: 요청 수 vs 걸린 시간")
print("=" * 78)
for n in [8, 50, 100, 300]:
    load_test(n)
    time.sleep(1)


  부하를 올려가며: 요청 수 vs 걸린 시간
     8개 | 전체   0.03초 | 개별 중앙   19.6ms  95%    21.9ms  최대    22.3ms | {200: 8}
    50개 | 전체   0.12초 | 개별 중앙   76.5ms  95%    92.6ms  최대    96.1ms | {200: 50}
   100개 | 전체   0.42초 | 개별 중앙  175.8ms  95%   363.7ms  최대   373.5ms | {200: 100}
   300개 | 전체   1.12초 | 개별 중앙  902.7ms  95%  1005.6ms  최대  1024.2ms | {200: 300}


결과는 **요청 37.5배(8 -> 300)에 시간 37.3배(0.03 -> 1.12초)** 였다.
정해둔 기준대로면 완전 직렬이다. 처리량으로 바꾸면 8개일 때나 300개일 때나 270 req/s 근처에서 평평했다.

그런데 여기서 측정 자체가 이상하다는 걸 알았다.
서버는 노트북 셀에서 띄운 것이라 **커널 안의 스레드**이고, 부하를 만드는 클라이언트 300스레드도
같은 커널의 스레드다. 둘이 GIL 하나를 나눠 쓰고 있으니
직렬로 나온 게 서버 때문인지 클라이언트 때문인지 가를 수가 없다.

#### B. 서버를 커널 밖으로 빼고 다시 쟀다

완전히 같은 `app/main_final.py` 를 별도 프로세스로 8001에 띄웠다.
클라이언트는 그대로 두고 상대만 바꿨으니, 달라지는 조건이 하나뿐이다.

In [36]:
# %load ../DP03/build/실험B_프로세스분리.py
# [내 실험 B] 서버를 커널 밖으로 빼고 다시 쟀다
#
# 실험 A 는 요청을 37.5배로 늘렸는데 시간도 37.3배로 늘었다. 완전 직렬이다.
# 그런데 그 측정에는 결함이 있었다. 서버가 노트북 커널 안의 스레드로 떠 있어서,
# 부하를 만드는 클라이언트 스레드들과 GIL 하나를 나눠 쓰고 있었다.
# 그러면 직렬로 나온 원인이 서버 때문인지 클라이언트 때문인지 가를 수가 없다.
#
# 그래서 완전히 같은 코드(app/main_final.py)를 별도 프로세스로 8001 에 하나 더 띄웠다.
# 클라이언트는 그대로 두고 상대만 바꿔서 재면, 달라지는 조건이 하나뿐이라 원인이 갈린다.

print("=" * 78)
print("  같은 코드 · 같은 클라이언트 · 서버 위치만 다름")
print("=" * 78)

for label, port in [("커널 안 (같은 프로세스)", 8000), ("커널 밖 (별도 프로세스)", 8001)]:
    print(f"\n[{label}]  http://localhost:{port}")
    for n in [8, 50, 100, 300]:
        load_test(n, url=f"http://localhost:{port}/predict/pixels")   # 실험 A 의 함수를 그대로 재사용


  같은 코드 · 같은 클라이언트 · 서버 위치만 다름

[커널 안 (같은 프로세스)]  http://localhost:8000
     8개 | 전체   0.03초 | 개별 중앙   17.6ms  95%    19.4ms  최대    19.9ms | {200: 8}
    50개 | 전체   0.12초 | 개별 중앙   75.6ms  95%    92.8ms  최대    95.6ms | {200: 50}
   100개 | 전체   0.22초 | 개별 중앙  146.1ms  95%   168.5ms  최대   174.1ms | {200: 100}
   300개 | 전체   1.13초 | 개별 중앙  906.1ms  95%  1014.2ms  최대  1024.4ms | {200: 300}

[커널 밖 (별도 프로세스)]  http://localhost:8001
     8개 | 전체   0.02초 | 개별 중앙   15.1ms  95%    15.9ms  최대    17.2ms | {200: 8}
    50개 | 전체   0.07초 | 개별 중앙   15.5ms  95%    24.0ms  최대    26.3ms | {200: 50}
   100개 | 전체   0.14초 | 개별 중앙   23.1ms  95%    30.1ms  최대    35.8ms | {200: 100}
   300개 | 전체   0.40초 | 개별 중앙   32.0ms  95%    41.1ms  최대    52.8ms | {200: 300}


코드는 한 글자도 안 바꿨는데 300개 기준 **1.13초에서 0.40초**로,
개별 대기는 **906밀리초에서 32밀리초**로 바뀌었다.
그러니까 A에서 본 완전 직렬은 서버의 성질이 아니라 측정 환경이 만든 것이었다.

노트북 6.2절이 시킨 방식, 그러니까 셀에서 서버를 띄우고 같은 셀에서 부하를 주는 배치는
실습으로 서버를 띄워보기엔 편한데 성능 숫자를 얻기에는 맞지 않는 것 같다.

#### C. run_in_executor 를 뺀 서버와 비교했다

여기까지 잰 건 executor 버전 하나뿐이라 비교 대상이 없었다.
그래서 `main_final.py` 를 복사해 `await loop.run_in_executor(...)` 한 줄만
`run_inference(...)` 로 바꾼 `app/main_blocking.py` 를 만들어 8002에 띄웠다.
나머지는 그대로 두었으니 달라지는 조건이 그 한 줄뿐이다.

서버 확인은 포트가 아니라 `openapi.json` 의 title 로 했다.
전날 포트만 보고 옛 서버를 재던 일이 있었다.

In [38]:
# %load ../DP03/build/실험C_executor대조.py
# [내 실험 C] run_in_executor 가 실제로 무슨 일을 하는지 — 대조군과 나란히
#
# 미션 4는 "run_in_executor 를 적용한 최종 서버를 동시 요청 테스트로 검증"하라고 했다.
# 그런데 지금까지 잰 건 executor 버전 하나뿐이라 비교 대상이 없었다.
# "750 req/s 나왔다"만으로는 그게 개선의 결과인지 원래 그런 건지 알 수가 없다.
#
# 그래서 main_final.py 를 복사하고 그 한 줄만 바꾼 대조군을 만들어 8002 에 띄웠다.
# 둘 다 커널 밖 별도 프로세스다 (실험 B 에서 커널 안에서 재면 못 믿는다는 걸 확인했으므로).

import requests

print("=" * 78)
print("  run_in_executor 있음 vs 없음 — 나머지 조건은 동일")
print("=" * 78)

# 어느 서버에 붙는지 포트가 아니라 서버가 자기 이름을 말하게 해서 확인한다.
# 어제 포트만 보고 옛 서버를 재던 사고가 있었다
for port in (8001, 8002):
    info = requests.get(f"http://localhost:{port}/openapi.json").json()["info"]
    print(f"  {port} -> {info['title']} ({info['version']})")

for label, port in [("run_in_executor 있음", 8001), ("없음 (async def 안에서 직접 호출)", 8002)]:
    print(f"\n[{label}]  http://localhost:{port}")
    for n in [8, 50, 100, 300]:
        total = load_test(n, url=f"http://localhost:{port}/predict/pixels")
        print(f"       └ 처리량 {n/total:6.0f} req/s")


  run_in_executor 있음 vs 없음 — 나머지 조건은 동일
  8001 -> MNIST Prediction API (3.0.0)
  8002 -> MNIST Prediction API (blocking 대조군) (3.0.0-blocking)

[run_in_executor 있음]  http://localhost:8001
     8개 | 전체   0.02초 | 개별 중앙   12.1ms  95%    15.9ms  최대    16.0ms | {200: 8}
       └ 처리량    413 req/s
    50개 | 전체   0.07초 | 개별 중앙   16.3ms  95%    22.6ms  최대    25.5ms | {200: 50}
       └ 처리량    680 req/s
   100개 | 전체   0.30초 | 개별 중앙  195.9ms  95%   202.3ms  최대   203.4ms | {200: 100}
       └ 처리량    331 req/s
   300개 | 전체   0.38초 | 개별 중앙   24.5ms  95%    34.2ms  최대    42.3ms | {200: 300}
       └ 처리량    785 req/s

[없음 (async def 안에서 직접 호출)]  http://localhost:8002
     8개 | 전체   0.02초 | 개별 중앙   11.3ms  95%    12.0ms  최대    13.5ms | {200: 8}
       └ 처리량    460 req/s
    50개 | 전체   0.07초 | 개별 중앙   17.1ms  95%    22.4ms  최대    25.2ms | {200: 50}
       └ 처리량    681 req/s
   100개 | 전체   0.14초 | 개별 중앙   22.5ms  95%    34.1ms  최대    51.0ms | {200: 100}
       └ 처리량    702 req/s
   300개 | 전체   0.42초 | 개별 

executor 785 req/s, 직접 호출 723 req/s 로 **차이가 8.6%** 나왔다.
그런데 같은 서버 안에서도 331에서 785까지 출렁였다.
잡음이 신호보다 훨씬 크니 차이가 있다고 말하기는 어렵고, 차이를 못 봤다고 적는 게 맞을 것 같다.

표 안에 이상한 데가 하나 더 있었다. 300개를 723/s로 처리한다면 맨 뒤 요청은 0.41초를 기다려야 하는데
실측 최대 대기가 43밀리초였다. 300개가 동시에 서버에 도착한 적이 없다는 뜻이라
부하를 그만큼 못 만들고 있었던 게 아닌가 싶었다.

#### D. 그래서 클라이언트가 낼 수 있는 최대치를 쟀다

서버가 거의 일을 안 하는 `/health` 를 같은 방식으로 때리면
거기서 나오는 숫자가 곧 내 측정 도구의 천장이다.

In [40]:
# %load ../DP03/build/실험D_클라이언트천장.py
# [내 실험 D] 723 req/s 는 서버의 한계인가, 내 측정 도구의 한계인가
#
# 실험 C 에서 executor 버전 785, 대조군 723 이 나왔다. 차이가 8.6% 인데
# 같은 서버 안에서도 331~785 로 출렁여서, 그 8.6% 를 차이라고 부를 수가 없다.
#
# 그 전에 더 근본적인 의심이 있다. 300개를 723/s 로 처리한다면 맨 뒤 사람은
# 0.41초를 기다려야 하는데 실측 최대 대기는 43.4ms 였다. 앞뒤가 안 맞는다.
# 300개가 동시에 도착한 적이 없다는 뜻이다 = 부하를 그만큼 못 만들고 있다.
#
# 그래서 서버가 거의 아무 일도 안 하는 /health 를 같은 방식으로 때려본다.
# 거기서 나오는 숫자가 곧 내 클라이언트가 낼 수 있는 최대치다.

import time
import logging
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed


def health_load(n_requests, port=8001):
    """서버가 일을 거의 안 하는 엔드포인트로 클라이언트 천장을 잰다"""
    url = f"http://localhost:{port}/health"
    log = logging.getLogger("ml_api")
    prev = log.level
    log.setLevel(logging.WARNING)

    def send(_):
        t0 = time.time()
        r = requests.get(url, timeout=60)
        return time.time() - t0, r.status_code

    try:
        start = time.time()
        with ThreadPoolExecutor(max_workers=n_requests) as ex:
            futures = [ex.submit(send, i) for i in range(n_requests)]
            results = [f.result() for f in as_completed(futures)]
        total = time.time() - start
    finally:
        log.setLevel(prev)

    times = sorted(r[0] for r in results)
    print(f"  {n_requests:>4}개 | 전체 {total:6.2f}초 | 처리량 {n_requests/total:6.0f} req/s | "
          f"중앙 {times[len(times)//2]*1000:6.1f}ms  최대 {times[-1]*1000:7.1f}ms")
    return n_requests / total


print("=" * 78)
print("  /health (서버가 거의 일 안 함) — 이 숫자가 클라이언트의 천장이다")
print("=" * 78)
ceilings = [health_load(n) for n in [50, 100, 300]]

print()
print("=" * 78)
print("  판정")
print("=" * 78)
best = max(ceilings)
print(f"  클라이언트가 낸 최대 부하 : {best:.0f} req/s")
print(f"  실험 C 의 추론 처리량     : executor 785 / blocking 723 req/s")
print()
if best > 1500:
    print("  -> 클라이언트는 여유가 있었다. 723 은 서버의 한계다.")
    print("     스레드풀 4개를 주고도 8% 밖에 못 번 것이므로 GIL 이 실제로 걸린 것.")
else:
    print("  -> 클라이언트가 그 근처에서 막힌다. 지금까지 잰 것은 서버가 아니라 내 측정 도구다.")
    print("     두 서버 중 어느 쪽이 나은지는 이 방법으로는 알 수 없다.")


  /health (서버가 거의 일 안 함) — 이 숫자가 클라이언트의 천장이다
    50개 | 전체   0.05초 | 처리량   1007 req/s | 중앙    8.0ms  최대    15.6ms
   100개 | 전체   0.09초 | 처리량   1161 req/s | 중앙    7.3ms  최대    17.7ms
   300개 | 전체   0.25초 | 처리량   1185 req/s | 중앙    7.6ms  최대    16.8ms

  판정
  클라이언트가 낸 최대 부하 : 1185 req/s
  실험 C 의 추론 처리량     : executor 785 / blocking 723 req/s

  -> 클라이언트가 그 근처에서 막힌다. 지금까지 잰 것은 서버가 아니라 내 측정 도구다.
     두 서버 중 어느 쪽이 나은지는 이 방법으로는 알 수 없다.


`/health` 는 **1185 req/s** 까지 나왔다. C의 서버들은 723~785이니 천장의 61~66%다.
코드에 미리 박아둔 판정 기준이 1500이었는데 하필 그 사이에 떨어져서 애매해졌다.
그 기준도 내가 감으로 정한 값이라 좋은 기준은 아니었던 것 같다.

원인은 프로세스는 갈랐는데 컴퓨터는 안 갈랐다는 데 있어 보인다.
GIL은 나뉘었지만 CPU 코어는 여전히 공유라 서버가 일하면 클라이언트 몫이 줄어든다.

여기서 방향을 바꿨다. 도구를 더 키우는 대신 **일을 무겁게** 하기로 했다.
추론 한 건이 몇백 밀리초면 클라이언트가 초당 몇 개만 만들어도 충분해서
병목이 확실하게 서버 쪽으로 넘어간다.

#### E. 추론을 로컬 LLM(ollama)으로 바꾸고 네 방식을 비교했다

다만 성격이 바뀐다. ollama는 별도 프로세스라 내 서버 입장에서 그 1.3초는
계산이 아니라 **남한테 시켜놓고 기다리는 시간**이다. CPU 바운드가 아니라 I/O 바운드다.
3장에서 쓴 `time.sleep(3)` 이 흉내내려던 게 이것이었다는 생각이 들었다.

미리 확인해보니 ollama는 요청을 하나씩 처리했다(3개 동시에 3.80초).
그러면 내 서버를 뭘로 짜든 전체 시간은 안 줄어든다.
그래서 비교하는 축을 처리량이 아니라 **추론이 도는 동안 헬스체크가 응답하는가**로 바꿨다.

In [42]:
# %load ../DP03/build/실험E_LLM네방식.py
# [내 실험 E] 추론을 별도 서버(ollama)로 넘겼을 때 — 네 가지 방식 비교
#
# MNIST 추론은 1.4ms 라 너무 가벼워서 차이가 안 보였다. 그래서 진짜 무거운 추론을 붙였다.
# 다만 ollama 는 별도 프로세스라, 내 서버 입장에서 그 1.3초는 계산이 아니라 기다림이다.
# 3장에서 쓴 time.sleep(3) 이 흉내내려던 게 바로 이것이다.
#
# ollama 는 요청을 하나씩 처리한다(3개 동시 = 3.80초, 미리 실측함).
# 그래서 내 서버를 어떻게 짜든 전체 시간은 안 줄어든다. 대신 갈리는 게 있다.
# 추론이 도는 동안 /health 가 응답하는가 — 느린 것과 죽은 것은 다르다.

import time
import threading
import requests

BASE = "http://localhost:8003"
METHODS = [
    ("blocking",   "① async def + 동기 HTTP"),
    ("threadpool", "② def (FastAPI 자동 스레드풀)"),
    ("executor",   "③ async def + run_in_executor"),
    ("async",      "④ async def + 비동기 클라이언트"),
]
N = 3          # 동시 추론 요청 수
HEALTH_AT = 0.5   # 추론이 돌기 시작하고 몇 초 뒤에 헬스체크를 던질지


def run(endpoint, n=N):
    """추론 n개를 동시에 보내고, 도중에 헬스체크를 한 번 던져 응답 시간을 잰다"""
    results = {}

    def infer(i):
        t0 = time.time()
        requests.post(f"{BASE}/llm/{endpoint}",
                      json={"prompt": f"숫자 {i} 에 대해 한 문장으로 설명해줘"}, timeout=300)
        results[f"infer{i}"] = time.time() - t0

    def health():
        time.sleep(HEALTH_AT)          # 추론이 자리를 잡은 뒤에 던진다
        t0 = time.time()
        requests.get(f"{BASE}/health", timeout=300)
        results["health"] = time.time() - t0

    threads = [threading.Thread(target=infer, args=(i,)) for i in range(n)]
    threads.append(threading.Thread(target=health))

    start = time.time()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    results["total"] = time.time() - start
    return results


print("=" * 82)
print(f"  추론 {N}개 동시 + 도중에 헬스체크 1개  (ollama qwen3:4b, 한 건당 약 1.3초)")
print("=" * 82)
print(f"  {'방식':<34} {'전체':>7} {'헬스체크':>9}   판정")
print("  " + "-" * 78)

for ep, label in METHODS:
    r = run(ep)
    verdict = "막힘" if r["health"] > 0.3 else "즉시 응답"
    print(f"  {label:<34} {r['total']:6.2f}초 {r['health']*1000:8.0f}ms   {verdict}")
    time.sleep(1)      # 서버와 ollama 가 다음 실험 전에 비워지도록

print()
print("  읽는 법")
print("   - 전체 시간이 넷 다 비슷하면: 천장은 내 서버가 아니라 ollama 다")
print("   - 헬스체크가 갈리면: 내 서버가 멈췄느냐 아니냐가 갈린 것")
print("   - 헬스체크 0.3초는 자의적 기준이 아니라, 추론 1건이 1.3초인데")
print("     그보다 훨씬 짧은 값을 '안 막혔다'로 본 것이다")


  추론 3개 동시 + 도중에 헬스체크 1개  (ollama qwen3:4b, 한 건당 약 1.3초)
  방식                                      전체      헬스체크   판정
  ------------------------------------------------------------------------------
  ① async def + 동기 HTTP                4.01초     3503ms   막힘
  ② def (FastAPI 자동 스레드풀)              3.80초        5ms   즉시 응답
  ③ async def + run_in_executor        3.80초        4ms   즉시 응답
  ④ async def + 비동기 클라이언트              3.84초        3ms   즉시 응답

  읽는 법
   - 전체 시간이 넷 다 비슷하면: 천장은 내 서버가 아니라 ollama 다
   - 헬스체크가 갈리면: 내 서버가 멈췄느냐 아니냐가 갈린 것
   - 헬스체크 0.3초는 자의적 기준이 아니라, 추론 1건이 1.3초인데
     그보다 훨씬 짧은 값을 '안 막혔다'로 본 것이다


전체 시간은 네 방식 모두 3.80~4.01초로 같았다. 천장이 ollama라는 게 그대로 보인다.
갈린 것은 헬스체크였다. **1번(async def + 동기 호출)은 3503밀리초, 나머지 셋은 3~5밀리초**다.

1번의 3503밀리초는 숫자가 딱 맞아떨어진다.
헬스체크를 0.5초에 던졌고 3.503초 뒤에 응답을 받았으니 합이 4.00초로, 전체 4.01초와 같다.
헬스체크가 추론 3개를 전부 기다린 것이다.

다만 2, 3, 4번 사이의 3, 4, 5밀리초는 차이가 아니라 잡음이라
이 실험으로는 셋을 가를 수 없었다. 셋이 갈리는 것은 시간이 아니라 스레드를 몇 개 쓰느냐일 것 같았다.

#### F. 서버가 스레드를 몇 개 쓰는지 세봤다

서버에 스레드 개수를 물어보는 것도 요청이라, 1번을 재는 동안에는 그 질문마저 막힌다.
그래서 이벤트 루프와 상관없는 별도 스레드가 0.05초마다 세어 쌓아두게 하고
부하가 끝난 뒤에 꺼내 보는 방식으로 짰다.

In [44]:
# %load ../DP03/build/실험F_스레드사용량.py
# [내 실험 F] ②③④ 는 시간으로 안 갈린다 — 갈리는 축은 스레드 사용량이다
#
# 실험 E 에서 헬스체크가 ① 3503ms vs ②③④ 3~5ms 로 갈렸다. 거기까진 명확하다.
# 그런데 ②③④ 사이의 3·4·5ms 는 차이가 아니라 잡음이다. 셋을 못 갈랐다.
#
# 못 가른 이유는 셋이 같아서가 아니라, 내가 잘못된 축으로 쟀기 때문이다.
# 셋은 "이벤트 루프를 안 막는다"는 결과가 같을 뿐 그 대가가 다르다.
#   ② 요청 하나당 스레드 하나 (FastAPI 기본 풀)
#   ③ 요청 하나당 스레드 하나 (내가 만든 풀, 4칸)
#   ④ 스레드 안 씀
# 그래서 시간이 아니라 스레드 개수를 센다. 서버가 스스로 0.05초마다 세어 쌓아두게 해뒀다.
#
# 마지막으로 글로벌 에러 핸들러가 실제로 걸리는 것도 확인한다.
# 6.3절에서는 200/422/400 만 나왔고 500 은 한 번도 안 걸렸다 — 만들어놓고 동작을 못 본 상태였다.

import time
import threading
import requests

BASE = "http://localhost:8003"
N = 8      # ③ 의 스레드풀이 4칸이므로, 그걸 넘겨야 한계가 보인다

print("=" * 88)
print(f"  추론 {N}개 동시 — 요청 하나를 처리하는 데 서버가 스레드를 몇 개 쓰나")
print("=" * 88)
print(f"  {'방식':<34} {'전체':>7} {'전체스레드':>10} {'내풀':>6} {'기본풀':>7}")
print("  " + "-" * 84)

for ep, label in [("blocking",   "① async def + 동기 HTTP"),
                  ("threadpool", "② def (FastAPI 자동 스레드풀)"),
                  ("executor",   "③ async def + run_in_executor"),
                  ("async",      "④ async def + 비동기 클라이언트")]:
    t0 = time.time()

    def infer(i):
        requests.post(f"{BASE}/llm/{ep}",
                      json={"prompt": f"숫자 {i} 에 대해 한 문장으로 설명해줘"}, timeout=300)

    ts = [threading.Thread(target=infer, args=(i,)) for i in range(N)]
    for t in ts:
        t.start()
    for t in ts:
        t.join()
    total = time.time() - t0

    # 부하가 끝난 뒤에 꺼내 본다. 재는 동안 물어보면 ①번에서는 그 질문도 막힌다
    d = requests.get(f"{BASE}/debug/threads", params={"since": t0}).json()
    print(f"  {label:<34} {total:6.2f}초 {d['전체최대']:9d} {d['inference풀최대']:6d} {d['AnyIO풀최대']:7d}")
    time.sleep(2)

print()
print("  평상시 스레드는 2개다 (메인 + 측정기). 거기서 얼마나 늘었는지를 본다.")

print()
print("=" * 88)
print("  글로벌 에러 핸들러 — 잡히지 않은 예외를 일부러 하나 터뜨린다")
print("=" * 88)
r = requests.get(f"{BASE}/debug/boom")
print(f"  클라이언트가 받은 응답 : HTTP {r.status_code}  {r.text}")
print(f"  서버가 살아 있는가     : {requests.get(f'{BASE}/health').json()}")
print()
print("  응답 어디에도 예외 종류·파일 경로·스택이 없다. 그건 서버 로그에만 남는다.")
print("  (터미널 로그: _scratch/06_Deployment/DP03/verify/uvicorn_8003.log)")


  추론 8개 동시 — 요청 하나를 처리하는 데 서버가 스레드를 몇 개 쓰나
  방식                                      전체      전체스레드     내풀     기본풀
  ------------------------------------------------------------------------------------
  ① async def + 동기 HTTP               10.64초         2      0       0
  ② def (FastAPI 자동 스레드풀)             10.02초        10      0       8
  ③ async def + run_in_executor        9.99초        14      4       8
  ④ async def + 비동기 클라이언트             10.14초        14      4       8

  평상시 스레드는 2개다 (메인 + 측정기). 거기서 얼마나 늘었는지를 본다.

  글로벌 에러 핸들러 — 잡히지 않은 예외를 일부러 하나 터뜨린다
  클라이언트가 받은 응답 : HTTP 500  {"success":false,"error":"서버 내부 오류가 발생했습니다."}
  서버가 살아 있는가     : {'status': 'healthy'}

  응답 어디에도 예외 종류·파일 경로·스택이 없다. 그건 서버 로그에만 남는다.
  (터미널 로그: _scratch/06_Deployment/DP03/verify/uvicorn_8003.log)


3번과 4번이 완전히 같은 숫자로 나왔다. 4번은 스레드를 안 쓴다고 했는데 이상했다.

원인은 스레드풀의 성질이었다. **스레드는 일이 끝나도 안 죽는다.** 다음에 또 쓰려고 살려둔다.
그래서 4번을 잴 때는 앞 실험들이 남긴 12개가 이미 깔려 있었고 내 측정기가 그걸 같이 셌다.
4번이 만든 게 아니라 물려받은 것이다. 측정이 문제였던 세 번째다.

실행 순서를 알면 증가분으로 읽을 수는 있다.
1번이 0개, 2번이 8개(요청 수와 같다), 3번이 4개(`max_workers` 와 같다), 4번이 0개다.

#### F2. 방식마다 서버를 새로 띄워서 다시 셌다

표 하나만 보고도 읽히게 하려고 다시 쟀다. 서버를 새로 띄우면 물려받을 스레드가 없다.
증가분으로 읽은 값이 맞다면 3번은 6 / 4 / 0, 4번은 2 / 0 / 0 으로 나와야 한다.

In [46]:
# %load ../DP03/build/실험F2_스레드_깨끗하게.py
# [내 실험 F-2] 실험 F 의 측정 결함을 고쳐서 다시 — 방식마다 서버를 새로 띄운다
#
# 실험 F 에서 ③과 ④가 완전히 같은 숫자로 나왔다. ④는 스레드를 안 쓴다고 했는데 이상했다.
# 원인은 스레드풀의 성질이었다. 스레드는 일이 끝나도 안 죽는다. 다음에 또 쓰려고 살려둔다.
# 그래서 ④를 잴 때는 ②③이 남긴 스레드 12개가 이미 깔려 있었고, 내 측정기는 그걸 같이 셌다.
# ④가 만든 게 아니라 물려받은 것이다.
#
# 실행 순서를 알면 증분으로 읽을 수는 있지만, 표 하나만 보고도 읽히게 하려고 다시 잰다.
# 방식마다 서버 프로세스를 새로 띄우면 물려받을 스레드가 없다.

import os
import time
import signal
import socket
import subprocess
import threading
import requests

PORT = 8003
BASE = f"http://localhost:{PORT}"
N = 8
ROOT = "/home/gmw/Documents/AIFFEL_Work/_scratch/06_Deployment/model-serving-course"
PY = f"{ROOT}/.venv_checkpoint/bin/python"


def port_busy(port):
    with socket.socket() as s:
        s.settimeout(0.3)
        return s.connect_ex(("127.0.0.1", port)) == 0


def fresh_server():
    """서버를 완전히 새로 띄운다. 앞 실험이 남긴 스레드가 없는 상태에서 시작하려는 것"""
    # 떠 있으면 먼저 내린다
    out = subprocess.run(["ss", "-ltnp"], capture_output=True, text=True).stdout
    for line in out.splitlines():
        if f":{PORT} " in line and "pid=" in line:
            pid = int(line.split("pid=")[1].split(",")[0])
            os.kill(pid, signal.SIGTERM)
    for _ in range(40):
        if not port_busy(PORT):
            break
        time.sleep(0.25)

    p = subprocess.Popen(
        [PY, "-m", "uvicorn", "app.main_llm:app", "--host", "127.0.0.1", "--port", str(PORT)],
        cwd=ROOT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    for _ in range(60):          # 뜰 때까지 기다린다. 포트가 아니라 /health 응답으로 판정
        try:
            if requests.get(f"{BASE}/health", timeout=1).status_code == 200:
                return p
        except Exception:
            pass
        time.sleep(0.5)
    raise RuntimeError("서버가 안 떴다")


print("=" * 88)
print(f"  추론 {N}개 동시 — 방식마다 서버를 새로 띄워서 잰다 (물려받은 스레드 0)")
print("=" * 88)
print(f"  {'방식':<34} {'전체':>7} {'전체스레드':>10} {'내풀':>6} {'기본풀':>7}")
print("  " + "-" * 84)

for ep, label in [("blocking",   "① async def + 동기 HTTP"),
                  ("threadpool", "② def (FastAPI 자동 스레드풀)"),
                  ("executor",   "③ async def + run_in_executor"),
                  ("async",      "④ async def + 비동기 클라이언트")]:
    proc = fresh_server()
    t0 = time.time()

    def infer(i):
        requests.post(f"{BASE}/llm/{ep}",
                      json={"prompt": f"숫자 {i} 에 대해 한 문장으로 설명해줘"}, timeout=300)

    ts = [threading.Thread(target=infer, args=(i,)) for i in range(N)]
    for t in ts:
        t.start()
    for t in ts:
        t.join()
    total = time.time() - t0

    d = requests.get(f"{BASE}/debug/threads", params={"since": t0}).json()
    print(f"  {label:<34} {total:6.2f}초 {d['전체최대']:9d} {d['inference풀최대']:6d} {d['AnyIO풀최대']:7d}")

print()
print("  평상시 2개(메인 + 측정기)에서 얼마나 늘었는지가 그 방식이 실제로 쓴 스레드다.")
print("  이제 실행 순서를 몰라도 표만 보고 읽을 수 있다.")

# 다음 실험을 위해 서버는 띄워둔 채로 끝낸다


  추론 8개 동시 — 방식마다 서버를 새로 띄워서 잰다 (물려받은 스레드 0)
  방식                                      전체      전체스레드     내풀     기본풀
  ------------------------------------------------------------------------------------
  ① async def + 동기 HTTP               17.21초         2      0       0
  ② def (FastAPI 자동 스레드풀)              9.98초        10      0       8
  ③ async def + run_in_executor        9.96초         6      4       0
  ④ async def + 비동기 클라이언트             10.02초         2      0       0

  평상시 2개(메인 + 측정기)에서 얼마나 늘었는지가 그 방식이 실제로 쓴 스레드다.
  이제 실행 순서를 몰라도 표만 보고 읽을 수 있다.


예상대로 나왔다. **3번은 6 / 4 / 0, 4번은 2 / 0 / 0** 이다.
평상시가 2개(메인과 측정기)이니 4번은 스레드를 하나도 안 늘리고 요청 8개를 처리한 것이다.
3번이 4에서 멈춘 것은 `max_workers=4` 그대로이고, 나머지 4개는 스레드를 못 받고 기다렸다는 뜻으로 보인다.

그런데 1번 전체 시간이 F에서는 10.64초였는데 여기서는 17.21초가 나왔다.
같은 조건에 62% 차이라 그냥 넘어갈 수가 없었다.

#### G. 그 숫자가 재현되는지 3번씩 다시 쟀다

이벤트 루프가 막혀서 다음 요청을 접수 못 한 시간이 샌 게 아닐까 싶었다.
가를 방법은 있었다. 엔드포인트가 서버 자신이 잰 시간을 응답에 담고 있는데 지금까지 그걸 버리고 있었다.
서버가 잰 시간의 합이 전체와 비슷하면 호출 안에서 다 쓴 것이고,
훨씬 작으면 호출 사이에서 샌 것이다.

4번을 같이 넣은 것은 대조군이 필요해서다. 1번만 흔들리는 건지 이 컴퓨터가 원래 그런 건지 갈라야 했다.

In [48]:
# %load ../DP03/build/실험G_시간이어디로샜나.py
# [내 실험 G] ①번의 7초는 어디로 샜나 — 그리고 17.21초는 재현되는 값인가
#
# 실험 F 에서 ①번이 10.64초, 실험 F-2 에서 같은 조건에 17.21초가 나왔다. 62% 차이다.
# ②③④ 는 두 번 다 10.0초 근처로 안정적이었는데 ①만 흔들렸다.
#
# 가설은 세울 수 있다. 이벤트 루프가 막혀 있으면 다음 요청을 접수조차 못 하니
# 그동안 ollama 가 놀았을 것이다 — 그럴듯하지만 그럴듯한 건 오늘 계속 틀렸다.
#
# 가르는 방법이 있다. 우리 엔드포인트는 서버가 직접 잰 시간(elapsed)을 응답에 담고 있는데
# 지금까지 그걸 버리고 있었다.
#   서버가 잰 시간의 합 ≈ 전체 시간   ->  ollama 호출 자체에서 다 쓴 것
#   서버가 잰 시간의 합 << 전체 시간   ->  호출과 호출 사이에서 샌 것 (접수를 못 한 시간)
#
# 그리고 3번씩 반복해서 17.21 이 재현되는 값인지 잡음인지도 같이 본다.

import time
import threading
import requests

if "fresh_server" not in globals():
    raise RuntimeError("실험 F-2 셀을 먼저 실행해라. fresh_server() 를 그대로 쓴다")

N = 8
REPS = 3

print("=" * 92)
print(f"  ①과 ④를 {REPS}번씩 — 전체 시간 vs 서버가 직접 잰 시간의 합")
print("=" * 92)
print(f"  {'방식':<26} {'회차':>4} {'전체':>8} {'서버시간합':>11} {'샌시간':>8} {'서버 개별(최소~최대)':>22}")
print("  " + "-" * 88)

summary = {}
for ep, label in [("blocking", "① async + 동기 HTTP"),
                  ("async",    "④ 비동기 클라이언트")]:
    totals = []
    for rep in range(1, REPS + 1):
        fresh_server()
        server_times = []
        lock = threading.Lock()

        def infer(i):
            r = requests.post(f"{BASE}/llm/{ep}",
                              json={"prompt": f"숫자 {i} 에 대해 한 문장으로 설명해줘"}, timeout=300)
            with lock:
                server_times.append(r.json()["elapsed"])

        t0 = time.time()
        ts = [threading.Thread(target=infer, args=(i,)) for i in range(N)]
        for t in ts:
            t.start()
        for t in ts:
            t.join()
        total = time.time() - t0
        totals.append(total)

        s = sum(server_times)
        print(f"  {label:<26} {rep:>4} {total:7.2f}초 {s:10.2f}초 {total - s:7.2f}초 "
              f"{min(server_times):>9.2f} ~ {max(server_times):.2f}초")
    summary[label] = totals

print()
print("=" * 92)
print("  판정")
print("=" * 92)
for label, totals in summary.items():
    spread = max(totals) - min(totals)
    print(f"  {label:<26} {[round(t, 2) for t in totals]}  폭 {spread:.2f}초")
print()
print("  ① 의 '샌시간' 이 크면: 이벤트 루프가 막혀서 다음 요청을 접수 못 한 시간이다.")
print("     그 동안 ollama 는 놀고 있었다는 뜻이라, 자원 낭비가 대기 시간 위에 얹힌다.")
print("  ① 의 '샌시간' 이 작으면: 내 가설이 틀렸고 다른 원인을 찾아야 한다.")
print("  폭(최대-최소)이 크면: 애초에 한 번 재고 비교하면 안 되는 숫자였다는 뜻이다.")


  ①과 ④를 3번씩 — 전체 시간 vs 서버가 직접 잰 시간의 합
  방식                           회차       전체       서버시간합      샌시간           서버 개별(최소~최대)
  ----------------------------------------------------------------------------------------
  ① async + 동기 HTTP             1   10.67초      10.63초    0.04초      1.32 ~ 1.34초
  ① async + 동기 HTTP             2   10.64초      10.62초    0.02초      1.32 ~ 1.33초
  ① async + 동기 HTTP             3   10.64초      10.63초    0.01초      1.32 ~ 1.33초
  ④ 비동기 클라이언트                   1   10.05초      45.30초  -35.25초      1.43 ~ 9.96초
  ④ 비동기 클라이언트                   2   10.10초      45.71초  -35.61초      1.48 ~ 10.07초
  ④ 비동기 클라이언트                   3   10.15초      46.02초  -35.87초      1.44 ~ 10.01초

  판정
  ① async + 동기 HTTP          [10.67, 10.64, 10.64]  폭 0.03초
  ④ 비동기 클라이언트                [10.05, 10.1, 10.15]  폭 0.11초

  ① 의 '샌시간' 이 크면: 이벤트 루프가 막혀서 다음 요청을 접수 못 한 시간이다.
     그 동안 ollama 는 놀고 있었다는 뜻이라, 자원 낭비가 대기 시간 위에 얹힌다.
  ① 의 '샌시간' 이 작으면: 내 가설이 틀렸고 다른 원인을 찾아야 한다.
  폭(최대-최소)이 크면: 애

**내 가설은 기각됐다.** 샌 시간이 0.01~0.04초로 거의 없었다.
그리고 1번을 3번 재니 10.67 / 10.64 / 10.64초로 폭이 0.03초였다.
17.21초는 재현되지 않았으니 이상치였던 것 같다.
모델이 새로 로딩된 게 아닌가도 생각해봤는데 `ollama ps` 를 보니 keep-alive가 아직 28분 남아 있어서
그것도 아니었다. **원인은 아직 모른다.**

한 번만 재고 1번이 7초 더 걸린다고 적었으면 틀린 숫자를 올릴 뻔했다.

4번에서는 더 재미있는 게 나왔다. 서버가 잰 시간의 합이 45.3초로 실제 걸린 10.05초의 4배가 넘는다.
내가 만든 지표가 음수를 뱉었는데, 그 고장 자체가 답을 알려준다.
개별 시간이 1번은 전부 1.32~1.34초로 똑같고, 4번은 1.43~9.96초로 뒤로 갈수록 늘어난다.
**줄이 서는 자리가 다른 것으로 보인다.**
1번은 요청 하나만 서버 안에 있고 나머지는 서버 밖에서 기다리니 서버가 자기가 밀렸다는 것도 모른다.
4번은 8개가 다 들어와 있어서 타임아웃이나 거절 같은 판단을 할 수 있는 상태가 된다.

전체 시간도 4번이 1번보다 0.6초(5.5%) 일관되게 빨랐고 3회 범위가 안 겹쳤다.
1번은 호출과 호출 사이에 서버 왕복이 끼어서 그동안 ollama가 잠깐씩 노는 것 같다.

#### H. 다시 CPU 바운드로 돌아와서 결론을 냈다

여기까지는 I/O 바운드 이야기라, 원래 궁금했던 것에는 답을 안 한 상태였다.
GIL 때문에 스레드가 CPU 작업에 소용없다고 배웠는데, 파이토치 추론에서도 정말 그런가.

조건을 세 가지 통제했다.
`torch.set_num_threads(1)` 로 한 요청이 코어 하나만 쓰게 묶고(이 기계는 32코어라 자리는 충분하다),
`repeat` 로 무게를 조절하고(순전파 1회가 0.302밀리초이고 1~400에서 선형인 걸 먼저 확인했다),
요청에 몸통을 없애서 A~D에서 병목이던 클라이언트 비용을 뺐다.

판정 지표도 하나로 정했다.
**병렬도 = (동시요청 4개 x 단건 시간) / 전체 시간** 이고, 1.0이면 완전 직렬, 4.0이면 완전 병렬이다.

In [50]:
# %load ../DP03/build/실험H_GIL결정실험.py
# [내 실험 H] 결정 실험 — 파이토치 추론에서 스레드풀은 진짜로 겹쳐 도는가
#
# 오늘 원래 물었던 질문이다. GIL 때문에 파이썬은 한 번에 한 스레드만 코드를 돌린다고 배웠다.
# 그렇다면 스레드를 4개 줘도 CPU 작업은 안 빨라져야 한다. 정말 그런가?
#
# 실험 C 는 MNIST 가 1.4ms 라 가벼워서 못 갈랐고, 실험 E~G 는 ollama 로 갔는데
# 그건 별도 프로세스라 I/O 바운드였다. CPU 바운드는 지금까지 미결이었다.
#
# 통제한 것 (app/main_cpu.py):
#   torch.set_num_threads(1)  한 요청 = 한 코어. 안 그러면 요청 하나가 32코어를 다 먹어서
#                             스레드를 줘도 자리가 없고, 그걸 GIL 탓으로 오해하게 된다
#   repeat 로 무게 조절        순전파 1회 = 0.302ms, 1~400 에서 선형임을 미리 확인
#   GET + 쿼리 (몸통 없음)     실험 A~D 에서 병목이던 클라이언트 JSON 비용을 없앤다
#
# 판정 지표 하나로 못 박는다:
#   병렬도 = (동시요청 4개 x 단건 시간) / 실제 걸린 시간
#     1.0 = 완전 직렬 (스레드가 겹쳐 돌지 않음)
#     4.0 = 완전 병렬 (스레드 4개가 진짜 동시에 돌았음)

import time
import statistics
import threading
import requests

BASE = "http://localhost:8004"
N = 4          # 스레드풀이 4칸이므로 4개로 맞춘다
REPS = 3       # 오늘 17.21초 이상치에 속을 뻔했다. 한 번 재고 결론내지 않는다
REPEATS = [1, 10, 100, 1000]      # 0.3ms / 3ms / 30ms / 302ms

METHODS = [("blocking",   "① async def + 직접 호출"),
           ("threadpool", "② def (자동 스레드풀)"),
           ("executor",   "③ run_in_executor (4칸)")]


def burst(ep, repeat, n=N):
    """동시 요청 n개를 보내고 전체 걸린 시간을 잰다"""
    def go(_):
        requests.get(f"{BASE}/cpu/{ep}", params={"repeat": repeat}, timeout=300)
    ts = [threading.Thread(target=go, args=(i,)) for i in range(n)]
    t0 = time.time()
    for t in ts:
        t.start()
    for t in ts:
        t.join()
    return time.time() - t0


results = {}
print("=" * 90)
print(f"  동시 요청 {N}개 · 방식마다 {REPS}회 반복 · torch 스레드 1개로 고정")
print("=" * 90)

for repeat in REPEATS:
    # 단건 시간을 먼저 잰다. 병렬도의 분자가 되는 값이라 매번 새로 잰다
    solo = statistics.median(
        [requests.get(f"{BASE}/cpu/blocking", params={"repeat": repeat}).json()["elapsed"]
         for _ in range(3)]
    )
    print(f"\n  repeat={repeat}  (단건 {solo*1000:.1f}ms)")
    for ep, label in METHODS:
        totals = [burst(ep, repeat) for _ in range(REPS)]
        med = statistics.median(totals)
        par = (N * solo) / med
        results[(repeat, ep)] = par
        print(f"    {label:<26} 전체 {med*1000:8.1f}ms   병렬도 {par:4.2f}   "
              f"(폭 {(max(totals)-min(totals))*1000:.1f}ms)")

print()
print("=" * 90)
print("  결론")
print("=" * 90)

heavy = REPEATS[-1]
par_block = results[(heavy, "blocking")]
par_pool = max(results[(heavy, "threadpool")], results[(heavy, "executor")])

print(f"  Q1. 파이토치 추론에서 스레드풀은 겹쳐 도는가?  (repeat={heavy} 기준)")
print(f"      직접 호출 병렬도 {par_block:.2f}  /  스레드풀 병렬도 {par_pool:.2f}")
if par_pool > 2.5:
    print(f"      -> 겹쳐 돈다. 파이토치가 계산 중 GIL 을 놓는다는 뜻이다.")
    print(f"         '스레드는 CPU 작업에 소용없다' 는 순수 파이썬 코드에 해당하는 말이고,")
    print(f"         C++ 로 내려가는 텐서 연산에는 해당하지 않는다.")
elif par_pool < 1.5:
    print(f"      -> 안 겹친다. GIL 이 그대로 걸려 있다. 배운 대로다.")
else:
    print(f"      -> 어중간하다. 부분적으로만 놓는다는 뜻이라 추가 조사가 필요하다.")

print()
print(f"  Q2. run_in_executor 는 작업이 얼마나 무거워야 값어치를 하는가?")
for repeat in REPEATS:
    ratio = results[(repeat, "executor")] / results[(repeat, "blocking")]
    ms = repeat * 0.302
    mark = "  <- 여기서부터 이득" if ratio > 1.5 else ""
    print(f"      추론 {ms:7.1f}ms : executor 가 직접호출 대비 {ratio:4.2f}배{mark}")


  동시 요청 4개 · 방식마다 3회 반복 · torch 스레드 1개로 고정

  repeat=1  (단건 0.8ms)
    ① async def + 직접 호출        전체      7.5ms   병렬도 0.43   (폭 0.6ms)
    ② def (자동 스레드풀)            전체      6.0ms   병렬도 0.53   (폭 1.4ms)
    ③ run_in_executor (4칸)     전체      5.0ms   병렬도 0.65   (폭 0.7ms)

  repeat=10  (단건 3.4ms)
    ① async def + 직접 호출        전체     17.7ms   병렬도 0.77   (폭 3.7ms)
    ② def (자동 스레드풀)            전체      9.5ms   병렬도 1.43   (폭 2.4ms)
    ③ run_in_executor (4칸)     전체      9.6ms   병렬도 1.41   (폭 0.7ms)

  repeat=100  (단건 30.5ms)
    ① async def + 직접 호출        전체    129.7ms   병렬도 0.94   (폭 3.6ms)
    ② def (자동 스레드풀)            전체     52.3ms   병렬도 2.33   (폭 3.5ms)
    ③ run_in_executor (4칸)     전체     53.7ms   병렬도 2.27   (폭 8.2ms)

  repeat=1000  (단건 301.9ms)
    ① async def + 직접 호출        전체   1221.6ms   병렬도 0.99   (폭 8.4ms)
    ② def (자동 스레드풀)            전체    366.2ms   병렬도 3.30   (폭 9.5ms)
    ③ run_in_executor (4칸)     전체    374.3ms   병렬도 3.23   (폭 22.3ms)

  결론
  Q1. 파이토치 추론에서 스레드풀은 겹쳐 도는가?

---

### 추가 실험 결과 정리

#### 1. 파이토치는 텐서 연산을 하는 동안 GIL 을 놓는 것으로 보인다

H의 병렬도가 이렇게 나왔다.

```
단건 시간    1번 직접호출   2번 def   3번 executor
  0.8ms        0.43        0.53       0.65
  3.4ms        0.77        1.43       1.41
 30.5ms        0.94        2.33       2.27
301.9ms        0.99        3.30       3.23
```

작업이 무거워질수록 1번은 1.0으로 수렴하고 2, 3번은 4.0 쪽으로 올라간다.
1번이 1.0이라는 건 4개를 보냈는데 1개 보낸 것과 같은 속도로만 처리했다는 뜻이다.

그러니까 "GIL 때문에 스레드는 CPU 작업에 소용없다"는 말은
**순수 파이썬 코드에 해당하는 이야기**이고, `model(x)` 처럼 C++ 로 내려가는 연산에는
그대로 적용되지 않는 것 같다. 미리 공부할 때 프로세스풀을 써야 GIL 을 피한다고 읽었는데,
넘파이나 파이토치 계산이라면 스레드로도 되는 조건이 붙는 것으로 보인다.

2, 3번이 4.0이 아니라 3.3에서 멈춘 것은
요청을 받고 응답을 만드는 일은 여전히 이벤트 루프 한 스레드가 순서대로 하기 때문이 아닐까 싶다.
계산 구간에서만 놓는 것이지 전부 놓는 건 아닌 것 같다.

가벼울 때(0.8밀리초) 병렬도가 0.43까지 내려간 것도 걸렸는데,
그건 추론보다 HTTP 처리가 더 커서 재고 있던 게 추론이 아니었던 것으로 생각된다.

#### 2. run_in_executor 에 손익분기점 같은 건 없어 보인다

처음엔 몇 밀리초부터 이득이 생기는지 찾으려 했는데, 그런 지점이 안 나왔다.
이득은 뚝 생기는 게 아니라 계속 커진다.

```
추론   0.3ms    7.5ms 가 5.0ms 로     2.5ms 절약
추론 302.0ms  1222ms 가  366ms 로    856ms 절약
```

비율로는 1.51배와 3.26배라 2배 남짓 차이인데, 실제로 아껴진 시간은 2.5밀리초와 856밀리초다.
그래서 비율이 아니라 **얼마나 아껴지는지**로 판단해야 하는 게 아닌가 싶다.
판정 코드에 1.5배라는 기준을 넣어놨더니 0.3밀리초짜리도 통과해버렸는데,
그 기준을 내가 감으로 정한 게 문제였다.

#### 3. 2번과 3번은 성능이 같고, 다른 건 한계를 누가 정하느냐다

병렬도가 3.30과 3.23이라 오차 범위 안이다. E에서도 헬스체크가 3밀리초와 4밀리초로 안 갈렸다.
F2에서 스레드 개수를 세보고 나서야 차이가 보였다.

```
방식                        루프    스레드(요청 8개)   한계를 정하는 것
1번 async def + 직접 호출    막힘         0            애초에 못 씀
2번 def (자동 스레드풀)       안 막힘       8            프레임워크 (40칸)
3번 run_in_executor         안 막힘       4            내가 정한 값
4번 비동기 클라이언트          안 막힘       0            없음
```

3번이 2번보다 빨라서 좋은 게 아니라 **칸 수를 내가 정할 수 있어서** 쓰는 것 같다.
추론 서버가 감당할 만큼으로 줄여두면 그쪽을 과부하시키지 않게 막을 수 있다.
4.6절 표를 이렇게 읽으면 되지 않을까 싶다.

#### 4. 추론을 별도 서버로 넘기면 성격이 바뀐다

ollama 처럼 추론을 다른 프로세스로 넘기면 내 서버 입장에서는 기다리는 시간이 되니
CPU 바운드가 아니라 I/O 바운드가 된다.
그때는 4번(비동기 클라이언트)이 스레드를 하나도 안 쓰고 같은 일을 했다.
미리 공부할 때 나온 Triton 이나 TorchServe 같은 별도 추론 서버 구조가
이런 모양이 아닐까 하는 생각이 들었다.

#### 5. 처리량과 응답성은 정하는 주체가 다른 것 같다

E에서 네 방식의 전체 시간이 3.80~4.01초로 다 같았다.
ollama 가 하나씩 처리하니 내 코드를 아무리 고쳐도 그 용량은 안 늘어난다.
그런데 헬스체크는 3503밀리초와 3밀리초로 갈렸다.

같은 3.8초를 쓰면서도 1번은 응답이 없는 서버이고 나머지는 일하는 중인 서버다.
로드밸런서가 헬스체크로 살았는지 판단한다고 배웠는데,
그러면 1번은 멀쩡히 일하다가 죽은 걸로 취급받아 트래픽이 끊길 수 있다.
느린 것과 죽은 것은 구분해야 한다는 게 여기서 와닿았다.

다만 처리량이 전혀 안 변하는 건 아니었다.
G에서 4번이 1번보다 0.6초(5.5%) 일관되게 빨랐는데,
1번은 호출 사이에 서버 왕복이 끼어 그동안 ollama 가 놀기 때문인 것 같다.
용량 자체는 못 늘려도 그 자원을 쉬지 않게 하는 정도는 내 쪽 몫으로 보인다.

#### 내가 틀렸던 것들

실험을 여덟 번 했는데 그중 상당수가 재려던 것보다 내 쪽이 문제였다.

```
측정 도구가 문제였던 것   4번   커널 공유 / 클라이언트 병목 / ollama 직렬 / 스레드 잔존
내 예상이 빗나간 것       3번   "직접호출은 166 req/s 일 것" / "시간이 샜을 것" / "파싱이 더 무거울 것"
판정 기준을 잘못 잡은 것   2번   1500 req/s (회색지대) / 1.5배 (전부 통과)
재현이 안 된 숫자         1번   17.21초, 다시 재니 10.64초
```

파싱이 무거워서 이득이 묻힌 게 아닐까 했던 가설은 재보니 바로 기각됐다.
파싱과 검증이 0.062밀리초, 추론이 0.306밀리초로 추론이 5배 무거웠다.
그럴듯해 보여도 재보기 전엔 모른다는 걸 여러 번 확인한 것 같다.

한 번만 재고 넘어갔으면 틀린 숫자가 몇 개는 들어갔을 것이다.
숫자 하나를 쓰기 전에 몇 번 재봤는지, 폭이 얼마인지를 같이 봐야겠다는 생각이 들었다.

#### 못 푼 것

F2에서 나온 17.21초의 원인은 결국 못 찾았다.
3회 재현에서 10.64초로 안정적이니 그쪽이 맞는 값이고 17.21이 이상치인 것까지는 확인했는데,
왜 그렇게 나왔는지는 모른다. 다른 프로세스가 끼어들었을 수도 있지만 확인은 못 했다.

---

### 추가 실험 I, J — 결론 3에는 잰 값이 없었다

위 정리를 써놓고 다시 읽어보니 결론 다섯 개 중 3번만 근거가 비어 있었다.
칸 수를 내가 정할 수 있어서 3번 방식을 쓴다고 적었는데,
정작 칸을 줄였을 때 줄 선 요청이 어떻게 되는지는 한 번도 안 쟀다.
재본 것과 추론한 것을 섞지 않겠다고 해놓고 그러지 못한 부분이라 마저 재보기로 했다.

여기까지의 노트북을 다른 사람들한테 보여주고 의견을 받았는데,
측정 도구 쪽으로 세 가지를 짚어줬다. 셋 다 앞 실험에 실제로 있던 문제라 여기서 같이 고쳤다.

```
1  localhost 대신 127.0.0.1     C 에서 100개가 전부 200ms 근처에 몰린 적이 있다.
                               분포가 뾰족하니 잡음이 아니라 고정 지연으로 보인다고 한다
2  스레드마다 Session 재사용      지금까지 요청마다 TCP 연결을 새로 맺고 있었다.
                               D 의 천장 1185 req/s 도 여기서 나왔을 수 있다
3  Barrier 로 동시 출발          ThreadPoolExecutor 에 submit 하면 스레드가 순차로 만들어진다.
                               300번째가 출발할 때 1번은 이미 끝나 있었다. 동시가 아니었다
```

3번이 특히 뼈아프다. A부터 D까지를 "300개 동시 요청"이라고 적었는데 실제로는 동시가 아니었다.

같이 받은 가설 중에 `localhost` 가 IPv6 로 먼저 풀려서 200ms 가 붙었을 거라는 게 있었는데,
이건 재보니 아니었다. 이 기계에서 `getent ahosts localhost` 는 127.0.0.1 만 내놓고,
`::1` 로 연결을 시도해도 0.202ms 만에 거부된다. 게다가 B 와 C 가 같은 주소를 썼는데
23.1ms 와 195.9ms 로 갈렸으니 주소 방식이 원인일 수가 없다.
**200ms 의 원인은 아직 모른다.** 다만 후보 하나를 미리 빼두는 것으로 정리했다.

병렬도의 분자와 분모도 손봤다. 실험 H 에서는 분자(단건 시간)를 서버가 잰 값으로,
분모(전체 시간)를 클라이언트가 잰 값으로 써서 시계가 서로 달랐다.
여기서는 둘 다 클라이언트 기준으로 맞췄다.

#### 가설을 미리 적어두고 시작한다

**H1. max_workers 는 처리량 천장이 아니라 줄 서는 구조를 정한다**

동시요청 N 이 칸 수 W 보다 크면 요청이 여러 파도로 나뉠 것 같았다.
그러면 p95 는 파도 수 곱하기 단건 시간이 될 것이다.
동시 16개, 단건 302밀리초 기준으로 숫자를 미리 박아두면 이렇다.

```
W=1   16파도   약 4.8초
W=2    8파도   약 2.4초
W=4    4파도   약 1.2초
W=8    2파도   약 0.6초
W=16   1파도   약 0.3초
```

**H2. 칸수 곱하기 torch 스레드 수가 코어 수를 넘으면 오히려 느려진다**

실험 H 에서 전제로 고정해둔 `torch.set_num_threads(1)` 을 이제 변수로 푼다.
이 기계는 32코어라 `(W=8, T=8)` 은 64코어를 요구하게 된다.

의견 준 쪽에서 `(W=32, T=1)` 을 32코어 조건으로 제안했는데 그건 아닌 것 같다.
동시요청이 8개면 칸을 32개 줘도 8개까지만 쓰이니 실제로는 8코어다.
그래서 성격을 바꿔서, 칸을 동시요청 수보다 늘려봐야 아무 일도 안 일어난다는 걸
확인하는 대조군으로 쓰기로 했다.

#### 0번 단계를 먼저 두는 이유

도구를 세 군데나 바꿨으니 그 상태에서 실험 H 와 같은 조건을 다시 재봐야 한다.
여기서 H 와 비슷한 값이 안 나오면 도구 수리가 뭔가를 바꾼 것이고,
그러면 H1 과 H2 결과도 믿을 수가 없다.

In [52]:
# %load ../DP03/build/실험I_max_workers와_코어.py
# [내 실험 I] 결론 3 만 근거가 없어서 — max_workers 를 실제로 돌려본다
#
# 결론 5개 중 3번만 재본 값이 없다. "3번 방식을 쓰는 이유는 칸 수를 내가 정할 수 있어서"라고 적었는데,
# 칸을 줄였을 때 줄 선 요청이 어떻게 되는지는 한 번도 안 쟀다.
# 내가 세운 기준(재본 것과 추론한 것을 섞지 않는다)에 비추면 미완이다.
#
# 앞선 실험들에서 지적받은 측정 도구 결함 세 가지를 여기서 같이 고친다.
#   1) localhost -> 127.0.0.1     실험 C 에서 100개가 전부 200ms 근처에 몰린 일이 있었다.
#                                 원인은 아직 모르지만 후보 하나는 미리 뺀다
#   2) 스레드마다 Session          지금까지는 요청마다 TCP 연결을 새로 맺었다.
#                                 실험 D 의 천장 1185 req/s 도 여기서 나왔을 수 있다
#   3) Barrier 로 동시 출발        ThreadPoolExecutor 에 submit 하면 스레드가 순차로 만들어진다.
#                                 300번째가 출발할 때 1번은 이미 끝나 있었다. 동시가 아니었다
#
# 가설 두 개를 미리 적어두고 시작한다.
#
# H1. max_workers 는 처리량 천장이 아니라 줄 서는 구조를 정한다
#     동시요청 N 이 칸 수 W 보다 크면 요청은 ceil(N/W) 개의 파도로 나뉜다.
#     예측: p95 = ceil(N/W) x 단건시간.  N=16, 단건 302ms 기준으로 미리 박아두면
#       W=1  16파도  약 4.8초 / W=2  8파도  약 2.4초 / W=4  4파도  약 1.2초
#       W=8   2파도  약 0.6초 / W=16 1파도  약 0.3초
#     예측보다 훨씬 크면 스레드풀 말고 다른 데서도 줄을 서고 있다는 뜻이다.
#
# H2. max_workers x torch_threads 가 코어 수를 넘으면 처리량이 떨어진다
#     이 기계는 32코어다. 실험 H 에서 전제로 고정해둔 torch 스레드를 이제 변수로 푼다.
#     (W=8, T=8) 은 64코어를 요구하니 (W=4, T=8) 보다 느려야 H2 가 산다.
#     ★ (W=32, T=1) 은 N=8 이면 워커가 8개까지만 쓰이므로 32코어 조건이 아니다.
#        칸을 N 보다 크게 늘려도 아무 일도 안 일어난다는 걸 보는 대조군으로 쓴다.

import os
import time
import math
import signal
import socket
import statistics
import subprocess
import threading
import requests

ROOT = "/home/gmw/Documents/AIFFEL_Work/_scratch/06_Deployment/model-serving-course"
PY = f"{ROOT}/.venv_checkpoint/bin/python"
PORT = 8004
BASE = f"http://127.0.0.1:{PORT}"       # 도구 수리 1
REPEAT = 1000                            # 단건 약 302ms
CORES = os.cpu_count()

_local = threading.local()


def session():
    """스레드마다 Session 하나. 커넥션을 재사용한다 (도구 수리 2)"""
    if not hasattr(_local, "s"):
        _local.s = requests.Session()
        _local.s.mount("http://", requests.adapters.HTTPAdapter(pool_maxsize=64))
    return _local.s


def fresh_server(pool, threads):
    """칸 수와 torch 스레드 수를 바꾸려면 서버를 새로 띄워야 한다"""
    out = subprocess.run(["ss", "-ltnp"], capture_output=True, text=True).stdout
    for line in out.splitlines():
        if f":{PORT} " in line and "pid=" in line:
            os.kill(int(line.split("pid=")[1].split(",")[0]), signal.SIGTERM)
    for _ in range(40):
        with socket.socket() as s:
            s.settimeout(0.3)
            if s.connect_ex(("127.0.0.1", PORT)) != 0:
                break
        time.sleep(0.25)

    env = {**os.environ, "POOL_SIZE": str(pool), "TORCH_THREADS": str(threads)}
    subprocess.Popen([PY, "-m", "uvicorn", "app.main_cpu:app", "--host", "127.0.0.1",
                      "--port", str(PORT)], cwd=ROOT, env=env,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try:
            h = requests.get(f"{BASE}/health", timeout=1).json()
            if h["pool"] == pool and h["torch_threads"] == threads:   # 포트가 아니라 값으로 확인한다
                return h
        except Exception:
            pass
        time.sleep(0.5)
    raise RuntimeError(f"서버가 안 떴거나 설정이 다르다 (pool={pool}, threads={threads})")


def solo():
    """지금 서버 설정에서 단건이 얼마나 걸리는지. 병렬도의 분자가 되는 값이다.
    실험 H 에서는 서버가 잰 시간을 썼는데 클라이언트 왕복 시간과 시계가 달랐다.
    여기서는 둘 다 클라이언트 기준으로 통일한다"""
    ts = []
    for _ in range(3):
        t0 = time.time()
        session().get(f"{BASE}/cpu/executor", params={"repeat": REPEAT}, timeout=300)
        ts.append(time.time() - t0)
    return statistics.median(ts)


def burst(n, endpoint="executor"):
    """n 개를 진짜로 동시에 출발시킨다 (도구 수리 3)"""
    barrier = threading.Barrier(n)
    lat, hdr, lock = [], [], threading.Lock()

    def go(_):
        s = session()
        s.get(f"{BASE}/health", timeout=30)      # 커넥션을 미리 맺어둔다. 핸드셰이크가 섞이지 않게
        barrier.wait()                           # 전원 대기 후 같이 출발
        t0 = time.time()
        r = s.get(f"{BASE}/cpu/{endpoint}", params={"repeat": REPEAT}, timeout=300)
        with lock:
            lat.append(time.time() - t0)
            hdr.append(float(r.headers.get("X-Process-Time", -1)))

    ths = [threading.Thread(target=go, args=(i,)) for i in range(n)]
    t0 = time.time()
    for t in ths:
        t.start()
    for t in ths:
        t.join()
    total = time.time() - t0
    lat.sort()
    return {"total": total, "p50": lat[len(lat) // 2], "p95": lat[int(len(lat) * 0.95) - 1],
            "max": lat[-1], "hdr_max": max(hdr)}


# ===== 0. 도구를 바꿨으니 먼저 실험 H 와 같은 값이 나오는지 확인한다 =====
print("=" * 92)
print(f"  0. 도구 수리 후 검증 — 실험 H 와 같은 조건(W=4, T=1, 동시 4개)을 다시 잰다   [{CORES}코어]")
print("=" * 92)
fresh_server(4, 1)
s0 = solo()
b0 = burst(4)
print(f"  단건 {s0*1000:6.1f}ms | 전체 {b0['total']*1000:7.1f}ms | 병렬도 {4*s0/b0['total']:4.2f}"
      f"   (실험 H 에서는 병렬도 3.23)")

# ===== H1 =====
N1 = 16
print()
print("=" * 92)
print(f"  H1. max_workers 가 줄 서는 구조를 정하는가   (동시 {N1}개 고정, torch 스레드 1개)")
print("=" * 92)
print(f"  {'칸수':>4} {'예상파도':>8} {'p95 예측':>10} {'p95 실측':>10} {'전체':>9} {'처리량':>10} {'헤더최대':>9}")
print("  " + "-" * 86)
h1 = {}
for W in [1, 2, 4, 8, 16]:
    fresh_server(W, 1)
    s = solo()
    runs = [burst(N1) for _ in range(3)]
    p95 = statistics.median(r["p95"] for r in runs)
    tot = statistics.median(r["total"] for r in runs)
    waves = math.ceil(N1 / W)
    h1[W] = (tot, p95, waves * s)
    print(f"  {W:>4} {waves:>8} {waves*s*1000:9.0f}ms {p95*1000:9.0f}ms {tot*1000:8.0f}ms "
          f"{N1/tot:9.1f}/s {max(r['hdr_max'] for r in runs)*1000:8.0f}ms")

# ===== H2 =====
N2 = 8
print()
print("=" * 92)
print(f"  H2. 칸수 x torch스레드 가 코어({CORES})를 넘으면 어떻게 되는가   (동시 {N2}개 고정)")
print("=" * 92)
print(f"  {'칸수':>4} {'T':>3} {'실제쓰는코어':>12} {'단건':>9} {'전체':>9} {'처리량':>10}")
print("  " + "-" * 86)
h2 = {}
for W, T in [(4, 1), (4, 8), (8, 4), (8, 8), (32, 1)]:
    fresh_server(W, T)
    s = solo()
    tot = statistics.median(burst(N2)["total"] for _ in range(3))
    used = min(W, N2) * T
    h2[(W, T)] = tot
    flag = "  <- 코어 초과" if used > CORES else ""
    print(f"  {W:>4} {T:>3} {used:>12} {s*1000:8.1f}ms {tot*1000:8.0f}ms {N2/tot:9.1f}/s{flag}")

# ===== 판정 =====
print()
print("=" * 92)
print("  판정")
print("=" * 92)
err = max(abs(h1[W][1] - h1[W][2]) / h1[W][2] for W in h1)
print(f"  H1: p95 예측 대비 최대 오차 {err*100:.0f}%")
print("      -> " + ("예측한 계단을 따라간다. 칸 수가 대기열 깊이를 정한 것으로 보인다."
                     if err < 0.35 else "예측에서 벗어난다. 스레드풀 말고 다른 데서도 줄을 서고 있다."))
print(f"  H1: 처리량 W=1 {N1/h1[1][0]:.1f}/s -> W=16 {N1/h1[16][0]:.1f}/s")

print(f"  H2: (4,8) {h2[(4,8)]*1000:.0f}ms  vs  (8,8) {h2[(8,8)]*1000:.0f}ms")
print("      -> " + ("코어를 넘겨 잡으면 느려진다. 오버서브스크립션이 실재한다."
                     if h2[(8, 8)] > h2[(4, 8)] * 1.1 else "느려지지 않는다. 이 크기 모델에서는 티가 안 난다."))
print(f"  H2: (4,8) {h2[(4,8)]*1000:.0f}ms  vs  (8,4) {h2[(8,4)]*1000:.0f}ms  (곱은 같고 배분만 다름)")
print(f"  H2: (8,1) 자리의 대조군 (32,1) = {h2[(32,1)]*1000:.0f}ms — 칸을 동시요청 수보다 늘려도 달라지는 게 없어야 한다")
print(f"  덤: X-Process-Time 최대값이 p95 근처면 미들웨어가 대기 시간까지 포함해 재는 것이다")


  0. 도구 수리 후 검증 — 실험 H 와 같은 조건(W=4, T=1, 동시 4개)을 다시 잰다   [32코어]
  단건  303.6ms | 전체   416.7ms | 병렬도 2.91   (실험 H 에서는 병렬도 3.23)

  H1. max_workers 가 줄 서는 구조를 정하는가   (동시 16개 고정, torch 스레드 1개)
    칸수     예상파도     p95 예측     p95 실측        전체        처리량      헤더최대
  --------------------------------------------------------------------------------------
     1       16      4852ms      4514ms     4839ms       3.3/s     4831ms
     2        8      2431ms      2475ms     2508ms       6.4/s     2513ms
     4        4      1209ms      1473ms     1501ms      10.7/s     1478ms
     8        2       608ms      1484ms     1521ms      10.5/s     1560ms
    16        1       303ms      2147ms     2167ms       7.4/s     2216ms

  H2. 칸수 x torch스레드 가 코어(32)를 넘으면 어떻게 되는가   (동시 8개 고정)
    칸수   T       실제쓰는코어        단건        전체        처리량
  --------------------------------------------------------------------------------------
     4   1            4    303.3ms      737ms      10.9/s
     4   8           32  

#### H1 은 기각됐다

```
칸수   예측 p95   실측 p95    오차
 1      4852       4514      -7%     맞음
 2      2431       2475      +2%     맞음
 4      1209       1473     +22%
 8       608       1484    +144%
16       303       2147    +608%     완전히 어긋남
```

W 가 1, 2 일 때는 파도 모델이 거의 맞는데 4부터 무너진다.
칸을 늘리면 파도 수가 줄어드니 계속 빨라질 거라고 봤는데, 그 전제가 틀렸다.

처리량을 보니 이유가 보인다.

```
W=1    3.3/s    1.00배
W=4   10.7/s    3.24배    <- 여기가 천장
W=8   10.5/s    3.18배
W=16   7.4/s    2.24배    <- 되레 나빠진다
```

**32코어인데 3.2배에서 멈추고, 16칸에서는 떨어진다.** 칸이 모자란 게 아니었다.

그래서 실험 H 에 적었던 추정 하나를 다시 봐야 할 것 같다.
그때 병렬도가 4.0 이 아니라 3.3 인 것을 두고
요청을 받고 응답을 만드는 일이 이벤트 루프에서 순차라서 그런 게 아닐까 적었는데,
HTTP 왕복은 302밀리초짜리 작업에 1밀리초 수준이라 3.2배 천장을 만들 수가 없다.
칸을 8, 16으로 늘려도 같은 천장이 나온 게 결정적으로 보인다.

천장의 모양도 걸린다. 작업의 일정 비율이 직렬이라 거기서 평평해지는 것이라면
올라가다 평평해져야지 떨어지면 안 된다. 떨어진다는 건
**스레드를 늘리는 것 자체에 비용이 있다**는 뜻이 아닐까 싶다.

#### H2 는 곱만 중요한 게 아니었다

```
(4, 8)   코어 32    975ms
(8, 4)   코어 32   1400ms    곱은 같은데 1.4배 느리다
```

배분도 변수인 것 같다. 그리고 T 를 올렸을 때 갈리는 게 하나 더 있다.

```
        단건        처리량
(4,1)   303.3ms    10.9/s
(4,8)   158.6ms     8.2/s
```

`torch.set_num_threads(8)` 로 연산 내부를 병렬화하면 **한 사람이 답 받는 시간은 절반**이 되는데
(303 에서 158밀리초) **전체 처리량은 떨어진다** (10.9 에서 8.2/s).
8배를 줬는데 1.9배만 빨라진 것도 같이 봐야 할 것 같다.

그러니까 다이얼이 두 개이고 방향이 반대다.
지연을 줄이고 싶으면 T 를 올리고, 처리량을 늘리고 싶으면 T 를 1로 두고 칸을 올린다.
이 모델 크기에서는 뒤쪽이 이겼다. `(32,1)` 이 691밀리초로 제일 좋았다.

대조군도 예상대로였다. `(32,1)` 은 동시요청이 8개뿐이라 칸 32개 중 8개만 쓰이는데,
그 값이 `(4,1)` 737밀리초와 비슷한 691밀리초다.
칸을 동시요청 수보다 늘리는 건 아무 일도 안 한다.

#### 덤으로 확인된 것

```
        헤더 최대   p95 실측
W=1      4831       4514
W=2      2513       2475
W=4      1478       1473
W=16     2216       2147
```

미들웨어가 붙여주는 `X-Process-Time` 이 p95 와 거의 같다.
**미들웨어는 스레드풀에서 줄 서서 기다린 시간까지 포함해서 재는 것으로 보인다.**
미리 공부할 때 미들웨어 입장에서는 스레드로 넘어갔다 온 것도 그냥 하나의 요청으로 보인다고
정리했었는데, 그게 숫자로 맞았다.

#### 내가 만든 오염 하나

0번 검증이 병렬도 2.91 로 나왔다. 실험 H 에서는 3.23 이었으니 10% 낮다.

도구를 고치면서 `X-Process-Time` 을 보려고 미들웨어도 같이 붙였는데
`BaseHTTPMiddleware` 는 공짜가 아니다. 조건 하나만 바꿔야 원인을 그 하나로 돌릴 수 있다고
계속 말해놓고 내가 둘을 바꿨다.
다만 실험 I 안에서의 비교는 전부 같은 서버 구성이라 위 결과들에는 영향이 없을 것 같다.

#### 그래서 하나만 더 — HTTP 를 통째로 빼본다

3.2배 천장이 파이토치 쪽인지 서버 쪽인지가 아직 안 갈렸다.
서버도 네트워크도 이벤트 루프도 없이 같은 작업만 스레드로 돌리면 그 자리에서 갈린다.

```
순수 파이썬도 3.2 근처에서 천장 + 16에서 하락    원인은 파이토치와 GIL. 서버는 무죄
순수 파이썬은 계속 올라간다                     원인은 서버 쪽이라 결론 1을 다시 봐야 한다
```

In [54]:
# %load ../DP03/build/실험J_HTTP를_빼고.py
# [내 실험 J] 3.2배 천장이 GIL 때문인가, 서버 때문인가
#
# 실험 I 에서 스레드를 4개보다 늘려도 3.2배가 천장이었고 16개에서는 오히려 떨어졌다.
# 32코어인데 그렇다. 원인 후보가 둘이다.
#   (가) 파이토치가 GIL 을 부분적으로만 놓아서. 그러면 서버는 무죄다
#   (나) FastAPI 나 HTTP 쪽에서 걸려서. 그러면 결론 1 을 다시 봐야 한다
#
# HTTP 를 통째로 빼고 같은 작업만 스레드로 돌리면 그 자리에서 갈린다.
# 서버도 없고 네트워크도 없고 이벤트 루프도 없다. 남는 건 파이토치와 스레드뿐이다.

import time
import threading
import statistics
import torch

torch.set_num_threads(1)          # 실험 I 와 같은 조건. 한 스레드 = 한 코어

import sys
sys.path.insert(0, "/home/gmw/Documents/AIFFEL_Work/_scratch/06_Deployment/model-serving-course")
from app.model_utils import load_model

MODEL = load_model("models/mnist_state_dict.pth")
SAMPLE = torch.randn(1, 1, 28, 28)
REPEAT = 1000


def work():
    """실험 I 의 heavy_inference 와 같은 일. HTTP 만 없다"""
    with torch.no_grad():
        for _ in range(REPEAT):
            out = MODEL(SAMPLE)
    return int(out.argmax().item())


for _ in range(3):                # 워밍업
    work()

solo = statistics.median([(lambda: (lambda t0: (work(), time.time() - t0)[1])(time.time()))() for _ in range(3)])

print("=" * 76)
print(f"  HTTP 없이 파이토치만 — 단건 {solo*1000:.1f}ms, 스레드 수를 늘려가며")
print("=" * 76)
print(f"  {'스레드':>6} {'전체':>10} {'배속':>7} {'예상(완전병렬)':>16}")
print("  " + "-" * 70)

for n in [1, 2, 4, 8, 16]:
    totals = []
    for _ in range(3):
        barrier = threading.Barrier(n)

        def go():
            barrier.wait()        # 전원 대기 후 같이 출발
            work()

        ths = [threading.Thread(target=go) for _ in range(n)]
        t0 = time.time()
        for t in ths:
            t.start()
        for t in ths:
            t.join()
        totals.append(time.time() - t0)
    med = statistics.median(totals)
    print(f"  {n:>6} {med*1000:9.0f}ms {n*solo/med:6.2f}배 {solo*1000:15.0f}ms")

print()
print("  배속이 3.2 근처에서 멈추고 16에서 떨어지면: 원인은 파이토치와 GIL 이고 서버는 무죄다.")
print("  배속이 계속 올라가면: 원인은 서버 쪽이라 결론 1 을 다시 봐야 한다.")


  HTTP 없이 파이토치만 — 단건 303.2ms, 스레드 수를 늘려가며
     스레드         전체      배속         예상(완전병렬)
  ----------------------------------------------------------------------
       1       304ms   1.00배             303ms
       2       314ms   1.93배             303ms
       4       376ms   3.22배             303ms
       8       737ms   3.29배             303ms
      16      2184ms   2.22배             303ms

  배속이 3.2 근처에서 멈추고 16에서 떨어지면: 원인은 파이토치와 GIL 이고 서버는 무죄다.
  배속이 계속 올라가면: 원인은 서버 쪽이라 결론 1 을 다시 봐야 한다.


#### 곡선이 거의 겹쳤다

```
스레드    실험 I (HTTP 포함)    실험 J (HTTP 없음)
  1          1.00배               1.00배
  2            -                  1.93배
  4          3.24배               3.22배
  8          3.18배               3.29배
 16          2.24배               2.22배
```

FastAPI 와 uvicorn 과 이벤트 루프와 `run_in_executor` 와 미들웨어를 전부 걷어냈는데
같은 곡선이 나왔다. 3.2에서 천장을 치고 16에서 무너지는 것은
서버가 만든 게 아니라 파이토치와 스레드 사이에서 벌어지는 일인 것 같다.

뒤집어 보면, 오늘 만든 서버는 파이토치가 낼 수 있는 최대를 그대로 통과시키고 있었다는 뜻이 된다.
서버 쪽에서 더 고칠 데가 안 보인다.

천장의 성격도 갈렸다. 작업의 일정 비율이 직렬이라 거기서 평평해지는 것이라면
2스레드 결과로 나머지를 예측할 수 있어야 한다.

```
2스레드 1.93배 에서 역산한 직렬 비율     약 3.6%
그 비율대로면 8스레드 예측              6.4배
실측                                  3.29배
```

예측이 두 배 가까이 빗나가고, 16에서는 아예 떨어진다.
고정된 직렬 구간이 있는 게 아니라 **스레드가 늘수록 GIL 을 주고받는 비용 자체가 커지는** 모양으로 보인다.
2개까지는 거의 공짜인데(1.93배) 4개부터 새기 시작하고 16개에서는 손해로 돌아선다.

#### 이게 4.5절과 맞부딪힌다

위 4.5절이 CPU 추론이면 코어 수와 비슷하게 잡으라고 안내했고,
그 셀을 이 기계에서 돌렸을 때 나온 출력이 이랬다.

```
CPU 코어 수: 32
권장 max_workers (CPU 추론): 32
```

그런데 실측으로는 16에서 이미 2.2배로 무너졌다. 32면 더 나쁠 것 같다.
이 작업에서 좋았던 값은 4에서 8 사이였다.

4.5절이 틀렸다기보다, 한계를 코어 수에 걸어둔 게 내 경우에는 안 맞는 것 같다.
4.5절도 너무 많으면 컨텍스트 스위칭 오버헤드가 생긴다고 같은 현상을 말하고 있는데,
그 한계가 코어 수보다 훨씬 먼저 왔다. 여기서는 코어의 8분의 1 지점이었다.
안 재보고 `os.cpu_count()` 를 그대로 넣었으면 네 배 가까이 잘못 잡을 뻔했다.

#### 정리하면

앞의 결론 다섯 개가 이렇게 바뀌고 하나가 늘었다.

```
1. 파이토치는 텐서 연산 중 GIL 을 놓는다 — 다만 전부는 아닌 것 같다
   스레드 2개까지는 거의 공짜(1.93배), 4~8개에서 3.2~3.3배로 천장,
   16개에서는 2.2배로 손해. 32코어여도 그렇다
   그리고 이 천장은 서버가 아니라 파이토치 쪽으로 보인다 (HTTP 를 다 빼도 같은 곡선)

2. run_in_executor 에 손익분기점은 없다. 판단은 비율이 아니라 아껴진 시간으로

3. 2번(def)과 3번(executor)은 성능이 같다. 다른 것은 한계를 누가 정하느냐다
   칸을 동시요청 수보다 늘리는 건 아무 일도 안 한다
   칸수와 torch 스레드는 곱만이 아니라 배분도 변수다

4. 추론을 별도 서버로 넘기면 CPU 바운드가 I/O 바운드가 된다. 그때는 4번이 스레드 0개

5. 처리량은 추론 자원이 정하고, 응답성은 내 서버가 정한다

6. 지연과 처리량은 서로 다른 다이얼이고 방향이 반대다
   그리고 max_workers 는 코어 수가 아니라 재서 정해야 하는 값인 것 같다
```

결론 3에 잰 값이 없다는 게 이 실험을 시작한 이유였는데, 그건 채워졌다.
대신 결론 1을 두 번 고치게 됐다. 처음에는 GIL 을 놓는다까지였고,
그다음에 전부는 아니라는 게 붙었고, 마지막에 그 한계가 서버 쪽이 아니라는 게 확인됐다.

#### 아직 못 푼 것

`localhost` 로 재던 구간에서 100개가 전부 200밀리초 근처에 몰렸던 일과,
같은 조건을 두 번 쟀는데 10.64초와 17.21초로 갈렸던 일은 원인을 못 찾았다.
뒤엣것은 세 번 다시 재서 10.64초가 맞는 값이라는 것까지는 확인했다.

#### 정리를 써놓고 보니 안 본 자리가 하나 남아 있었다

미션 4를 다시 읽어보면 "예외가 발생해도 서버가 죽지 않고 안전한 에러 응답을 반환하는 구조"를 만들라고 되어 있다.
6.3절에서 200, 422, 400 은 봤고, 앞에서 일부러 예외를 내서 500 도 한 번 봤다.

그런데 그 500 은 **이벤트 루프에서 터진 예외**였다.
오늘 만든 구조에서 추론이 실제로 도는 자리는 `run_in_executor` 로 넘긴 **스레드풀 안**이다.
거기서 터지면 어떻게 되는지는 한 번도 안 봤다.

물음이 두 개인데 두 번째가 더 걸린다.

```
1  스레드풀 워커 안에서 터진 예외가 글로벌 핸들러까지 와서 500 이 되는가
2  그 워커 스레드가 죽어서 풀이 4칸에서 3칸으로 줄지는 않는가  (이쪽이 더 걸린다)
```

2번이 사실이라면, 에러가 날 때마다 서버 처리 능력이 조금씩 깎이는데
겉으로 보이는 건 500 뿐이다. 응답은 정상적으로 나가고 로그도 남으니
한동안 아무도 모르다가 어느 순간 느려져 있는 상태가 된다.
에러 없이 조용히 나빠지는 종류라 한 번은 확인해두는 게 좋겠다고 생각했다.

예외가 터질 수 있는 자리를 세 군데 만들어 각각 10번씩 터뜨리고,
앞뒤로 스레드 수를 세어봤다. 스레드를 세는 장치는 실험 F 에서 만든 것을 그대로 쓴다.

In [56]:
# %load ../DP03/build/실험K_예외가어디서터지나.py
# [내 실험 K] 예외가 스레드풀 안에서 터지면 어떻게 되나
#
# 미션 4는 "예외가 발생해도 서버가 죽지 않고 안전한 에러 응답을 반환하는 구조"를 만들라고 했다.
# 그런데 앞에서 확인한 건 /debug/boom 하나뿐이고, 그건 이벤트 루프에서 터진 예외다.
# 오늘 만든 구조에서 추론이 실제로 도는 자리는 스레드풀 안이다. 거기는 아직 안 봤다.
#
# 물음 둘
#   1) 스레드풀 워커 안에서 터진 예외가 글로벌 핸들러까지 와서 500 이 되는가
#   2) ★그 워커 스레드가 죽어서 풀이 4칸에서 3칸으로 줄지는 않는가
#
# 2번이 고약하다. 그렇다면 에러가 날 때마다 서버 처리 능력이 조금씩 깎이는데
# 겉으로는 500 만 보인다. 에러 없이 조용히 나빠지는 종류다.

import time
import requests

BASE = "http://127.0.0.1:8004"
s = requests.Session()


def threads(label):
    d = s.get(f"{BASE}/debug/threads").json()
    print(f"  {label:<34} 전체 {d['전체']:>3} | inference풀 {d['inference풀']} | AnyIO풀 {d['AnyIO풀']}")
    return d


print("=" * 84)
print("  1단계 · 풀을 데운다 (추론을 4개 동시에 보내 워커 4개를 만든다)")
print("=" * 84)
threads("시작 전")

from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(lambda _: s.get(f"{BASE}/cpu/executor", params={"repeat": 200}), range(4)))
before = threads("추론 4개 뒤 (풀이 찼다)")

print()
print("=" * 84)
print("  2단계 · 예외를 세 자리에서 각각 10번씩 터뜨린다")
print("=" * 84)
print(f"  {'터지는 자리':<30} {'상태코드':>10} {'응답 본문'}")
print("  " + "-" * 80)

for ep, label in [("boom-in-loop",   "이벤트 루프에서"),
                  ("boom-in-def",    "기본 스레드풀에서 (def)"),
                  ("boom-in-thread", "내 전용 스레드풀에서")]:
    codes, bodies = [], set()
    for _ in range(10):
        r = s.get(f"{BASE}/debug/{ep}")
        codes.append(r.status_code)
        bodies.add(r.text[:60])
    uniq = sorted(set(codes))
    print(f"  {label:<30} {str(uniq):>10} {list(bodies)[0]}")

print()
after = threads("예외 30번 뒤")

print()
print("=" * 84)
print("  판정")
print("=" * 84)
d_inf = before["inference풀"] - after["inference풀"]
d_any = before["AnyIO풀"] - after["AnyIO풀"]
print(f"  inference풀 {before['inference풀']} -> {after['inference풀']}   (차이 {d_inf})")
print(f"  AnyIO풀     {before['AnyIO풀']} -> {after['AnyIO풀']}   (차이 {d_any})")
print()
if d_inf == 0 and d_any <= 0:
    print("  -> 워커가 안 죽는다. 예외를 낸 스레드는 그대로 다음 작업을 받는다.")
    print("     ThreadPoolExecutor 가 작업을 try 로 감싸 예외를 Future 에 담아 돌려주기 때문으로 보인다.")
else:
    print("  -> ★풀이 줄었다. 에러가 날 때마다 처리 능력이 깎인다는 뜻이라 심각한 문제다.")

print()
print("  3단계 · 예외를 30번 낸 뒤에도 서버가 멀쩡한지")
t0 = time.time()
r = s.get(f"{BASE}/cpu/executor", params={"repeat": 1000})
print(f"    추론 요청  HTTP {r.status_code}  {(time.time()-t0)*1000:.0f}ms   {r.json().get('elapsed')}초")
print(f"    헬스체크   {s.get(f'{BASE}/health').json()}")


  1단계 · 풀을 데운다 (추론을 4개 동시에 보내 워커 4개를 만든다)
  시작 전                               전체   1 | inference풀 0 | AnyIO풀 0
  추론 4개 뒤 (풀이 찼다)                    전체   5 | inference풀 4 | AnyIO풀 0

  2단계 · 예외를 세 자리에서 각각 10번씩 터뜨린다
  터지는 자리                               상태코드 응답 본문
  --------------------------------------------------------------------------------
  이벤트 루프에서                            [500] {"success":false,"error":"서버 내부 오류가 발생했습니다."}
  기본 스레드풀에서 (def)                     [500] {"success":false,"error":"서버 내부 오류가 발생했습니다."}
  내 전용 스레드풀에서                         [500] {"success":false,"error":"서버 내부 오류가 발생했습니다."}

  예외 30번 뒤                           전체   6 | inference풀 4 | AnyIO풀 1

  판정
  inference풀 4 -> 4   (차이 0)
  AnyIO풀     0 -> 1   (차이 -1)

  -> 워커가 안 죽는다. 예외를 낸 스레드는 그대로 다음 작업을 받는다.
     ThreadPoolExecutor 가 작업을 try 로 감싸 예외를 Future 에 담아 돌려주기 때문으로 보인다.

  3단계 · 예외를 30번 낸 뒤에도 서버가 멀쩡한지
    추론 요청  HTTP 200  302ms   0.2997초
    헬스체크   {'status': 'healthy', 'torch_threads': 1, 'pool': 4}

#### 워커는 안 죽었다

```
터지는 자리                 상태코드   응답 본문
이벤트 루프에서               500     {"success":false,"error":"서버 내부 오류가 발생했습니다."}
기본 스레드풀에서 (def)        500     같음
내 전용 스레드풀에서           500     같음
```

**세 자리 모두 500 이고 본문도 똑같다.** 예외가 어디서 터졌는지는 클라이언트 쪽에서 구분되지 않는다.
글로벌 핸들러가 스레드 안에서 터진 예외까지 받아낸다는 뜻으로 보인다.

스레드 수는 이렇게 나왔다.

```
              예외 전   예외 30번 뒤
inference풀      4          4
AnyIO풀          0          1
```

`inference풀` 이 그대로 4다. 워커가 안 죽었다.

`AnyIO풀` 이 0에서 1로 는 것이 오히려 좋은 증거인 것 같다.
`boom-in-def` 는 일반 `def` 라 FastAPI 기본 스레드풀에서 도는데,
**10번을 터뜨렸는데 워커가 1개만 생겼다.** 한 워커가 열 번을 다 받아냈다는 뜻이다.
매번 죽었다면 새로 만들어지면서 숫자가 올라갔을 것이다.

이유는 `ThreadPoolExecutor` 의 동작 방식에 있는 것 같다.
워커가 작업을 부를 때 그 호출을 감싸두고, 예외가 나면 그걸 잡아서 `Future` 에 담아 돌려준다.
그러니 예외는 작업의 결과값처럼 전달되고 워커 자신은 멀쩡히 다음 작업을 받는다.

예외를 서른 번 낸 뒤에도 추론 요청이 200 으로 302ms 에 돌아왔고 헬스체크도 정상이었다.
미션 4가 요구한 "예외가 발생해도 서버가 죽지 않는 구조"는 이걸로 확인된 것 같다.

한 가지 조심할 것은, 여기서 본 게 **예외가 잡힌다는 것까지**라는 점이다.
예외가 터지기 전에 그 워커가 잡아둔 자원, 예를 들어 열어둔 파일이나 GPU 메모리 같은 것이
제대로 정리되는지는 이 실험으로 알 수 없다. 그건 다른 이야기이고 오늘은 안 봤다.

---

### 6.4 프로젝트 구조 확인

```
model-serving-course/
├── 📁 app/
│   ├── error_handlers.py          ← Day 3: 글로벌 에러 핸들러
│   ├── logger_config.py           ← Day 3: 로깅 설정
│   ├── main_final.py              ← Day 3: 최종 서버
│   ├── middleware.py              ← Day 3: 요청/응답 로깅 미들웨어
│   ├── model_utils.py             ← Day 1: 모델 유틸리티
│   └── schemas.py                 ← Day 2: Pydantic 스키마
├── 📁 models/
│   └── mnist_state_dict.pth
├── .gitignore
└── requirements.txt
```

---

### ✅ Day 3 최종 체크포인트

```
Q1. 동기 서버에서 3초 걸리는 추론을 3명이 동시에 요청하면 총 몇 초 걸립니까?
Q2. time.sleep(3)과 await asyncio.sleep(3)의 핵심 차이는?
Q3. async def 안에서 동기 블로킹 코드를 실행하면 왜 헬스체크까지 영향받습니까?
Q4. run_in_executor가 이벤트 루프 블로킹을 방지하는 원리는?
Q5. 글로벌 Exception Handler를 사용하는 이유는?
Q6. 클라이언트에게 스택 트레이스를 노출하면 안 되는 이유는?
```

---

### 📌 Day 3 요약

```
오늘 한 일:
  ✅ 동기/비동기의 차이와 모델 추론에서의 의미를 이해했습니다.
  ✅ 동기 추론이 서버를 멈추는 문제를 재현하고 헬스체크 차단까지 관찰했습니다.
  ✅ run_in_executor 패턴으로 문제를 해결했습니다.
  ✅ 글로벌 에러 핸들러와 로깅으로 서버 안정성을 높였습니다.
  ✅ 동시 요청 테스트로 동작을 확인했습니다.

내일 (Day 4):
  🔜 Streamlit으로 웹 UI를 만들고 FastAPI와 연결합니다.
```

### 제출

다음 내역을 MD 파일로 기록, 깃헙에 업로드하여 링크로 제출하시기 바랍니다  

1. 각 섹션 실행 스샷
2. 각 섹션 체크포인트의 답변

수고하셨습니다!